In [1]:
# 10_pre_sleep_forecasting_dataset_design.ipynb
# Cell 1. pre-sleep forecasting dataset 설계를 위한 source 파일/컬럼 탐색
# 원하는 결과:
# - 수면 episode의 start/end datetime이 들어 있을 가능성이 있는 processed 파일을 찾는다.
# - 기존 modeling dataset과 sleep target 관련 파일의 컬럼을 확인한다.
# - 아직 dataset 생성이나 training은 하지 않는다.

from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

# sleep target / modeling / daily feature 관련 후보 파일을 넓게 탐색합니다.
candidate_patterns = [
    "*sleep*.csv",
    "*target*.csv",
    "*modeling*.csv",
    "*encoded*.csv",
    "*daily*.csv",
    "*merge*.csv",
]

candidate_files = []
for pattern in candidate_patterns:
    candidate_files.extend(PROCESSED_DIR.rglob(pattern))

candidate_files = sorted(set(candidate_files))

file_rows = []
for path in candidate_files:
    try:
        # 큰 파일도 처음 몇 줄만 읽어 컬럼과 shape 일부를 확인합니다.
        preview = pd.read_csv(path, encoding="utf-8-sig", nrows=5)
        columns = list(preview.columns)

        time_like_columns = [
            col for col in columns
            if any(token in col.lower() for token in ["time", "date", "start", "end", "sleep"])
        ]

        file_rows.append(
            {
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "exists": path.exists(),
                "size_mb": path.stat().st_size / (1024 ** 2),
                "preview_rows": len(preview),
                "columns_count": len(columns),
                "time_or_sleep_like_columns": time_like_columns,
            }
        )
    except Exception as exc:
        file_rows.append(
            {
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "exists": path.exists(),
                "size_mb": path.stat().st_size / (1024 ** 2) if path.exists() else np.nan,
                "preview_rows": None,
                "columns_count": None,
                "time_or_sleep_like_columns": f"READ_ERROR: {exc}",
            }
        )

candidate_file_summary = pd.DataFrame(file_rows)

print("\n=== Candidate Processed CSV Files ===")
display(candidate_file_summary)

# 우선순위가 높은 기존 파일들을 직접 확인합니다.
priority_paths = [
    PROCESSED_DIR / "sleep_target_table.csv",
    PROCESSED_DIR / "modeling_dataset_missing_handled.csv",
    PROCESSED_DIR / "modeling_dataset_encoded.csv",
    PROCESSED_DIR / "deep_learning_previous_day" / "modeling_dataset_previous_day_encoded.csv",
]

priority_rows = []
priority_previews = {}

for path in priority_paths:
    row = {
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
    }

    if path.exists():
        df_preview = pd.read_csv(path, encoding="utf-8-sig", nrows=10)
        priority_previews[str(path.relative_to(PROJECT_ROOT))] = df_preview

        columns = list(df_preview.columns)
        row.update(
            {
                "columns_count": len(columns),
                "columns": columns,
                "time_or_sleep_like_columns": [
                    col for col in columns
                    if any(token in col.lower() for token in ["time", "date", "start", "end", "sleep"])
                ],
            }
        )
    else:
        row.update(
            {
                "columns_count": None,
                "columns": None,
                "time_or_sleep_like_columns": None,
            }
        )

    priority_rows.append(row)

priority_summary = pd.DataFrame(priority_rows)

print("\n=== Priority File Column Summary ===")
display(priority_summary[["relative_path", "exists", "columns_count", "time_or_sleep_like_columns"]])

for rel_path, preview_df in priority_previews.items():
    print(f"\n=== Preview: {rel_path} ===")
    display(preview_df.head(5))

# start/end datetime 후보 컬럼을 더 구체적으로 찾습니다.
datetime_candidate_rows = []

for path in candidate_files:
    try:
        df_preview = pd.read_csv(path, encoding="utf-8-sig", nrows=20)
    except Exception:
        continue

    for col in df_preview.columns:
        col_lower = col.lower()
        if any(token in col_lower for token in ["start", "end", "bed", "wake", "sleep", "time", "date"]):
            sample_values = (
                df_preview[col]
                .dropna()
                .astype(str)
                .head(5)
                .tolist()
            )

            datetime_candidate_rows.append(
                {
                    "relative_path": str(path.relative_to(PROJECT_ROOT)),
                    "column": col,
                    "dtype_preview": str(df_preview[col].dtype),
                    "sample_values": sample_values,
                }
            )

datetime_candidate_table = pd.DataFrame(datetime_candidate_rows)

print("\n=== Datetime / Sleep Column Candidates ===")
display(datetime_candidate_table)

print("\nNext step:")
print("- Identify which file has sleep episode start/end datetime.")
print("- Then verify whether good_sleep_label can be aligned to sleep_start_datetime.")

PROJECT_ROOT: c:\workSpace\DeepLearnin_sleep
PROJECT_ROOT exists: True

=== Candidate Processed CSV Files ===


,relative_path,exists,size_mb,preview_rows,columns_count,time_or_sleep_like_columns
0,data\processed\daily_aggregates\fitbit_activity_minutes_daily.csv,True,0.334445,5,6,[calendar_date]
1,data\processed\daily_aggregates\fitbit_calories_daily.csv,True,0.326138,5,4,[calendar_date]
2,data\processed\daily_aggregates\fitbit_daily_hrv_summary_daily.csv,True,0.138433,5,6,[calendar_date]
3,data\processed\daily_aggregates\fitbit_daily_spo2_daily.csv,True,0.065606,5,6,[calendar_date]
4,data\processed\daily_aggregates\fitbit_hrv_details_daily.csv,True,0.602516,5,19,[calendar_date]
5,data\processed\daily_aggregates\fitbit_respiratory_rate_summary_daily.csv,True,0.158854,5,9,"[calendar_date, respiratory_full_sleep_breathing_rate_mean, respiratory_full_sleep_signal_to_noise_mean, respiratory_full_sleep_standard_deviation_mean, res..."
6,data\processed\daily_aggregates\fitbit_resting_heart_rate_daily.csv,True,0.660950,5,5,[calendar_date]
7,data\processed\daily_aggregates\fitbit_steps_daily.csv,True,0.219474,5,4,[calendar_date]
8,data\processed\daily_aggregates\fitbit_stress_daily.csv,True,0.118944,5,9,"[calendar_date, stress_sleep_points_mean]"
9,data\processed\daily_aggregates\fitbit_wrist_temperature_daily.csv,True,0.278324,5,6,[calendar_date]



=== Priority File Column Summary ===


,relative_path,exists,columns_count,time_or_sleep_like_columns
0,data\processed\sleep_target_table.csv,False,NaN,None
1,data\processed\modeling_dataset_missing_handled.csv,True,200.0,"[calendar_date, good_sleep_label, respiratory_full_sleep_breathing_rate_mean_missing_ind, respiratory_full_sleep_breathing_rate_mean, respiratory_full_sleep..."
2,data\processed\modeling_dataset_encoded.csv,True,200.0,"[calendar_date, good_sleep_label, respiratory_full_sleep_breathing_rate_mean_missing_ind, respiratory_full_sleep_breathing_rate_mean, respiratory_full_sleep..."
3,data\processed\deep_learning_previous_day\modeling_dataset_previous_day_encoded.csv,True,203.0,"[calendar_date, feature_date, good_sleep_label, respiratory_full_sleep_breathing_rate_mean_missing_ind, respiratory_full_sleep_breathing_rate_mean, respirat..."



=== Preview: data\processed\modeling_dataset_missing_handled.csv ===


...


=== Preview: data\processed\deep_learning_previous_day\modeling_dataset_previous_day_encoded.csv ===


,participant_object_id,calendar_date,feature_date,feature_lag_days,good_sleep_label,deep_learning_split,stress_score_mean_missing_ind,stress_score_mean,stress_responsiveness_points_mean_missing_ind,stress_responsiveness_points_mean,stress_exertion_points_mean_missing_ind,stress_exertion_points_mean,stress_ready_rate_missing_ind,stress_ready_rate,stress_calculation_failed_rate_missing_ind,stress_calculation_failed_rate,stress_record_count_missing_ind,stress_record_count,hrv_summary_rmssd_mean_missing_ind,hrv_summary_rmssd_mean,hrv_summary_nremhr_mean_missing_ind,hrv_summary_nremhr_mean,hrv_summary_entropy_mean_missing_ind,hrv_summary_entropy_mean,hrv_summary_record_count_missing_ind,hrv_summary_record_count,hrv_detail_rmssd_mean_missing_ind,hrv_detail_rmssd_mean,hrv_detail_rmssd_std_missing_ind,hrv_detail_rmssd_std,hrv_detail_rmssd_min_missing_ind,hrv_detail_rmssd_min,hrv_detail_rmssd_max_missing_ind,hrv_detail_rmssd_max,hrv_detail_coverage_mean_missing_ind,hrv_detail_coverage_mean,hrv_detail_coverage_std_missing_ind,hrv_detail_coverage_std,hrv_detail_coverage_min_missing_ind,hrv_detail_coverage_min,hrv_detail_coverage_max_missing_ind,hrv_detail_coverage_max,hrv_detail_low_frequency_mean_missing_ind,hrv_detail_low_frequency_mean,hrv_detail_low_frequency_std_missing_ind,hrv_detail_low_frequency_std,hrv_detail_low_frequency_min_missing_ind,hrv_detail_low_frequency_min,hrv_detail_low_frequency_max_missing_ind,hrv_detail_low_frequency_max,hrv_detail_high_frequency_mean_missing_ind,hrv_detail_high_frequency_mean,hrv_detail_high_frequency_std_missing_ind,hrv_detail_high_frequency_std,hrv_detail_high_frequency_min_missing_ind,hrv_detail_high_frequency_min,hrv_detail_high_frequency_max_missing_ind,hrv_detail_high_frequency_max,hrv_detail_record_count_missing_ind,hrv_detail_record_count,...,place_outdoors_count_missing_ind,place_outdoors_count,place_transit_count_missing_ind,place_transit_count,place_work_school_count_missing_ind,place_work_school_count,mood_alert_rate_missing_ind,mood_alert_rate,mood_anger_rate_missing_ind,mood_anger_rate,mood_fear_rate_missing_ind,mood_fear_rate,mood_happy_rate_missing_ind,mood_happy_rate,mood_joy_rate_missing_ind,mood_joy_rate,mood_neutral_rate_missing_ind,mood_neutral_rate,mood_rested_relaxed_rate_missing_ind,mood_rested_relaxed_rate,mood_sad_rate_missing_ind,mood_sad_rate,mood_sadness_rate_missing_ind,mood_sadness_rate,mood_surprise_rate_missing_ind,mood_surprise_rate,mood_tense_anxious_rate_missing_ind,mood_tense_anxious_rate,mood_tired_rate_missing_ind,mood_tired_rate,place_entertainment_rate_missing_ind,place_entertainment_rate,place_gym_rate_missing_ind,place_gym_rate,place_home_rate_missing_ind,place_home_rate,place_home_office_rate_missing_ind,place_home_office_rate,place_other_rate_missing_ind,place_other_rate,place_outdoors_rate_missing_ind,place_outdoors_rate,place_transit_rate_missing_ind,place_transit_rate,place_work_school_rate_missing_ind,place_work_school_rate,survey_response_count_missing_ind,survey_response_count,survey_bfpt_count_missing_ind,survey_bfpt_count,survey_breq_count_missing_ind,survey_breq_count,survey_dq_count_missing_ind,survey_dq_count,survey_panas_count_missing_ind,survey_panas_count,survey_stai_count_missing_ind,survey_stai_count,survey_ttmspbf_count_missing_ind,survey_ttmspbf_count
0,621e2e8e67b776a24055b564,2021-05-25,2021-05-24,1.0,1,train,0.0,78.0,0.0,26.0,0.0,27.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,89.603,0.0,57.432,0.0,3.148,0.0,1.0,0.0,94.334853,0.0,18.218702,0.0,51.154,0.0,137.764,0.0,0.960608,0.0,0.053802,0.0,0.742,0.0,1.006,0.0,2894.455108,0.0,1694.483510,0.0,584.193,0.0,11050.486,0.0,2385.181873,0.0,1070.098448,0.0,360.402,0.0,5481.430,0.0,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,17.0,0.0,1.0,0.0,2.0,0.0,1.0,0.0,6.0,0.0,6.0,0.0,1.0
1,621e2e8e67b776a24055b564,2021-05-26,2021-05-25,1.0,1,train,0.0,80.0,0.0


=== Datetime / Sleep Column Candidates ===


,relative_path,column,dtype_preview,sample_values
0,data\processed\daily_aggregates\fitbit_activity_minutes_daily.csv,calendar_date,object,"[2021-05-24, 2021-05-25, 2021-05-26, 2021-05-27, 2021-05-28]"
1,data\processed\daily_aggregates\fitbit_calories_daily.csv,calendar_date,object,"[2021-05-24, 2021-05-25, 2021-05-26, 2021-05-27, 2021-05-28]"
2,data\processed\daily_aggregates\fitbit_daily_hrv_summary_daily.csv,calendar_date,object,"[2021-05-24, 2021-05-25, 2021-05-26, 2021-05-27, 2021-05-28]"
3,data\processed\daily_aggregates\fitbit_daily_spo2_daily.csv,calendar_date,object,"[2021-10-19, 2021-05-24, 2021-05-25, 2021-05-26, 2021-05-27]"
4,data\processed\daily_aggregates\fitbit_hrv_details_daily.csv,calendar_date,object,"[2021-05-24, 2021-05-25, 2021-05-26, 2021-05-27, 2021-05-28]"
5,data\processed\daily_aggregates\fitbit_respiratory_rate_summary_daily.csv,calendar_date,object,"[2021-05-24, 2021-05-25, 2021-05-26, 2021-05-27, 2021-05-28]"
6,data\processed\daily_aggregates\fitbit_respiratory_rate_summary_daily.csv,respiratory_full_sleep_breathing_rate_mean,float64,"[14.8, 15.8, 14.6, 14.8, 15.2]"
7,data\processed\daily_aggregates\fitbit_respiratory_rate_summary_daily.csv,respiratory_full_sleep_signal_to_noise_mean,float64,"[12.022, 8.119, 8.677, 13.04, 6.696]"
8,data\processed\daily_aggregates\fitbit_respiratory_rate_summary_daily.csv,respiratory_full_sleep_standard_deviation_mean,float64,"[1.1, 0.9, 1.1, 0.9, 1.6]"
9,data\processed\daily_aggregates\fitbit_respiratory_rate_summary_daily.csv,respiratory_deep_sleep_breathing_rate_mean,float64,"[14.6, 14.6, 14.6, 14.6, 14.2]"



Next step:
- Identify which file has sleep episode start/end datetime.
- Then verify whether good_sleep_label can be aligned to sleep_start_datetime.


In [2]:
# Cell 2. sleep episode start/end datetime 정밀 점검
# 원하는 결과:
# - sleep_daily_target.csv와 modeling_dataset_daily.csv의 sleep start/end 구조를 확인한다.
# - calendar_date가 start date인지 end date인지 검증한다.
# - cross-midnight sleep 비율을 확인한다.
# - pre-sleep forecasting의 target episode index를 만들 수 있는지 판단한다.
# - 아직 파일 저장은 하지 않는다.

SLEEP_TARGET_PATH = PROCESSED_DIR / "daily_aggregates" / "sleep_daily_target.csv"
MODELING_DAILY_PATH = PROCESSED_DIR / "modeling_dataset_daily.csv"

sleep_target = pd.read_csv(SLEEP_TARGET_PATH, encoding="utf-8-sig")
modeling_daily = pd.read_csv(MODELING_DAILY_PATH, encoding="utf-8-sig")

for frame in [sleep_target, modeling_daily]:
    frame["calendar_date"] = pd.to_datetime(frame["calendar_date"], errors="coerce")
    frame["startTime"] = pd.to_datetime(frame["startTime"], errors="coerce")
    frame["endTime"] = pd.to_datetime(frame["endTime"], errors="coerce")

sleep_target["sleep_start_date"] = sleep_target["startTime"].dt.normalize()
sleep_target["sleep_end_date"] = sleep_target["endTime"].dt.normalize()
sleep_target["calendar_date_dt"] = sleep_target["calendar_date"].dt.normalize()
sleep_target["sleep_duration_from_times_hours"] = (
    sleep_target["endTime"] - sleep_target["startTime"]
).dt.total_seconds() / 3600

sleep_target["calendar_matches_start_date"] = (
    sleep_target["calendar_date_dt"] == sleep_target["sleep_start_date"]
)
sleep_target["calendar_matches_end_date"] = (
    sleep_target["calendar_date_dt"] == sleep_target["sleep_end_date"]
)
sleep_target["cross_midnight"] = (
    sleep_target["sleep_start_date"] != sleep_target["sleep_end_date"]
)

print("=== Sleep Target Shape ===")
print("sleep_target:", sleep_target.shape)
print("modeling_daily:", modeling_daily.shape)

print("\n=== Sleep Target Required Column Check ===")
required_columns = [
    "participant_object_id",
    "calendar_date",
    "startTime",
    "endTime",
    "good_sleep_label",
]
required_check = pd.DataFrame(
    {
        "column": required_columns,
        "in_sleep_target": [col in sleep_target.columns for col in required_columns],
        "in_modeling_daily": [col in modeling_daily.columns for col in required_columns],
    }
)
display(required_check)

print("\n=== Sleep Start/End Missingness ===")
missing_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "participants",
            "calendar_date_missing",
            "startTime_missing",
            "endTime_missing",
            "good_sleep_label_missing",
        ],
        "value": [
            len(sleep_target),
            sleep_target["participant_object_id"].nunique(),
            int(sleep_target["calendar_date"].isna().sum()),
            int(sleep_target["startTime"].isna().sum()),
            int(sleep_target["endTime"].isna().sum()),
            int(sleep_target["good_sleep_label"].isna().sum()),
        ],
    }
)
display(missing_summary)

print("\n=== Calendar Date Meaning Check ===")
calendar_meaning_summary = pd.DataFrame(
    {
        "metric": [
            "calendar_matches_start_date_rows",
            "calendar_matches_end_date_rows",
            "cross_midnight_rows",
            "cross_midnight_ratio",
            "calendar_matches_start_date_ratio",
            "calendar_matches_end_date_ratio",
        ],
        "value": [
            int(sleep_target["calendar_matches_start_date"].sum()),
            int(sleep_target["calendar_matches_end_date"].sum()),
            int(sleep_target["cross_midnight"].sum()),
            float(sleep_target["cross_midnight"].mean()),
            float(sleep_target["calendar_matches_start_date"].mean()),
            float(sleep_target["calendar_matches_end_date"].mean()),
        ],
    }
)
display(calendar_meaning_summary)

print("\n=== Sleep Duration Sanity Check ===")
duration_summary = sleep_target[
    ["minutesAsleep", "timeInBed", "sleep_duration_hours", "time_in_bed_hours", "sleep_duration_from_times_hours"]
].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
display(duration_summary)

print("\n=== Target Distribution ===")
target_distribution = (
    sleep_target["good_sleep_label"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("good_sleep_label")
    .reset_index(name="rows")
)
target_distribution["ratio"] = target_distribution["rows"] / target_distribution["rows"].sum()
display(target_distribution)

print("\n=== Duplicate Episode Key Check ===")
# start/end time이 있으면 participant + startTime + endTime이 episode key 후보입니다.
duplicate_key_count = int(
    sleep_target.duplicated(
        ["participant_object_id", "startTime", "endTime"],
        keep=False,
    ).sum()
)

duplicate_calendar_count = int(
    sleep_target.duplicated(
        ["participant_object_id", "calendar_date"],
        keep=False,
    ).sum()
)

duplicate_summary = pd.DataFrame(
    {
        "key": [
            "participant + startTime + endTime",
            "participant + calendar_date",
        ],
        "duplicated_rows": [
            duplicate_key_count,
            duplicate_calendar_count,
        ],
    }
)
display(duplicate_summary)

if duplicate_key_count > 0:
    print("\nDuplicate participant/start/end examples:")
    display(
        sleep_target[
            sleep_target.duplicated(["participant_object_id", "startTime", "endTime"], keep=False)
        ].sort_values(["participant_object_id", "startTime"]).head(20)
    )

if duplicate_calendar_count > 0:
    print("\nDuplicate participant/calendar_date examples:")
    display(
        sleep_target[
            sleep_target.duplicated(["participant_object_id", "calendar_date"], keep=False)
        ].sort_values(["participant_object_id", "calendar_date", "startTime"]).head(20)
    )

print("\n=== Example Sleep Episodes ===")
display(
    sleep_target[
        [
            "participant_object_id",
            "calendar_date",
            "startTime",
            "endTime",
            "sleep_start_date",
            "sleep_end_date",
            "cross_midnight",
            "minutesAsleep",
            "timeInBed",
            "good_sleep_label",
        ]
    ].head(20)
)

print("\n=== Preliminary Design Implication ===")
if sleep_target["calendar_matches_end_date"].mean() > 0.95:
    print("calendar_date is effectively sleep_end_date. Pre-sleep forecasting should realign features to sleep_start_date/startTime.")
elif sleep_target["calendar_matches_start_date"].mean() > 0.95:
    print("calendar_date is effectively sleep_start_date. Same-date daily features may be closer to pre-sleep date, but bedtime cutoff still matters.")
else:
    print("calendar_date has mixed meaning. Use explicit startTime/endTime for episode-level alignment.")

=== Sleep Target Shape ===
sleep_target: (3551, 35)
modeling_daily: (3551, 130)

=== Sleep Target Required Column Check ===


,column,in_sleep_target,in_modeling_daily
0,participant_object_id,True,True
1,calendar_date,True,True
2,startTime,True,True
3,endTime,True,True
4,good_sleep_label,True,True



=== Sleep Start/End Missingness ===


,metric,value
0,rows,3551
1,participants,69
2,calendar_date_missing,0
3,startTime_missing,0
4,endTime_missing,0
5,good_sleep_label_missing,0



=== Calendar Date Meaning Check ===


...


=== Duplicate Episode Key Check ===


,key,duplicated_rows
0,participant + startTime + endTime,0
1,participant + calendar_date,0



=== Example Sleep Episodes ===


,participant_object_id,calendar_date,startTime,endTime,sleep_start_date,sleep_end_date,cross_midnight,minutesAsleep,timeInBed,good_sleep_label
0,621e2e8e67b776a24055b564,2021-05-24,2021-05-24 00:40:00,2021-05-24 09:21:00,2021-05-24,2021-05-24,False,445,521,1
1,621e2e8e67b776a24055b564,2021-05-25,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,2021-05-25,True,460,548,1
2,621e2e8e67b776a24055b564,2021-05-26,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,2021-05-26,True,493,560,1
3,621e2e8e67b776a24055b564,2021-05-27,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,2021-05-27,True,540,627,1
4,621e2e8e67b776a24055b564,2021-05-28,2021-05-27 23:37:00,2021-05-28 08:58:30,2021-05-27,2021-05-28,True,493,561,1
5,621e2e8e67b776a24055b564,2021-05-29,2021-05-28 23:50:00,2021-05-29 08:48:00,2021-05-28,2021-05-29,True,471,538,1
6,621e2e8e67b776a24055b564,2021-05-30,2021-05-30 00:30:00,2021-05-30 09:45:30,2021-05-30,2021-05-30,False,485,555,1
7,621e2e8e67b776a24055b564,2021-05-31,2021-05-30 23:29:30,2021-05-31 09:08:30,2021-05-30,2021-05-31,True,497,579,1
8,621e2e8e67b776a24055b564,2021-06-01,2021-05-31 23:42:30,2021-06-01 09:26:30,2021-05-31,2021-06-01,True,503,584,1
9,621e2e8e67b776a24055b564,2021-06-02,2021-06-01 23:58:30,2021-06-02 09:32:30,2021-06-01,2021-06-02,True,479,574,1



=== Preliminary Design Implication ===
calendar_date is effectively sleep_end_date. Pre-sleep forecasting should realign features to sleep_start_date/startTime.


In [3]:
# Cell 3. sleep_start_date 기준 pre-sleep feature alignment 가능성 점검
# 원하는 결과:
# - sleep episode별 prediction_time = startTime 직전으로 정의한다.
# - 기존 daily feature table을 sleep_start_date에 붙일 수 있는지 확인한다.
# - 기존 previous_day feature_date가 sleep_start_date와 얼마나 일치했는지 확인한다.
# - daytime_only feature coverage를 start-date 기준으로 확인한다.
# - 아직 파일 저장은 하지 않는다.

ID_COL = "participant_object_id"
TARGET_COL = "good_sleep_label"

DAYTIME_ONLY_FEATURE_PATH = PROCESSED_DIR / "deep_learning_feature_subsets" / "daytime_only_features.csv"
PREVIOUS_DAY_DATA_PATH = PROCESSED_DIR / "deep_learning_previous_day" / "modeling_dataset_previous_day_encoded.csv"

daytime_only_features = pd.read_csv(DAYTIME_ONLY_FEATURE_PATH, encoding="utf-8-sig")["feature"].tolist()
previous_day_df = pd.read_csv(PREVIOUS_DAY_DATA_PATH, encoding="utf-8-sig")

previous_day_df["calendar_date"] = pd.to_datetime(previous_day_df["calendar_date"], errors="coerce")
previous_day_df["feature_date"] = pd.to_datetime(previous_day_df["feature_date"], errors="coerce")

# Episode index: 하나의 sleep episode가 하나의 prediction target입니다.
episode_index = sleep_target.copy()
episode_index["sleep_episode_id"] = (
    episode_index[ID_COL].astype(str)
    + "__"
    + episode_index["startTime"].dt.strftime("%Y%m%d%H%M%S")
    + "__"
    + episode_index["endTime"].dt.strftime("%Y%m%d%H%M%S")
)

episode_index["sleep_start_date"] = episode_index["startTime"].dt.normalize()
episode_index["sleep_end_date"] = episode_index["endTime"].dt.normalize()
episode_index["prediction_time"] = episode_index["startTime"]

episode_columns = [
    ID_COL,
    "sleep_episode_id",
    "calendar_date",
    "startTime",
    "endTime",
    "sleep_start_date",
    "sleep_end_date",
    "prediction_time",
    TARGET_COL,
    "minutesAsleep",
    "timeInBed",
    "sleep_duration_hours",
    "time_in_bed_hours",
]

episode_index = episode_index[episode_columns].copy()

# daily feature source 후보:
# modeling_dataset_daily는 start/end/target과 daily aggregates가 함께 들어 있습니다.
# pre-sleep 설계에서는 sleep outcome 관련 컬럼을 feature에서 제외하고,
# participant + calendar_date를 daily feature key로만 사용합니다.
daily_feature_source = modeling_daily.copy()
daily_feature_source["calendar_date"] = pd.to_datetime(
    daily_feature_source["calendar_date"],
    errors="coerce",
).dt.normalize()

sleep_outcome_or_id_columns = [
    ID_COL,
    "calendar_date",
    "startTime",
    "endTime",
    "minutesAsleep",
    "minutesAwake",
    "timeInBed",
    "sleep_duration_hours",
    "time_in_bed_hours",
    "awake_ratio",
    "wake_minutes",
    "wake_ratio",
    "asleep_minutes",
    "awake_minutes",
    "classic_asleep_ratio",
    "classic_awake_ratio",
    TARGET_COL,
]

available_daily_feature_columns = [
    col for col in daily_feature_source.columns
    if col not in sleep_outcome_or_id_columns
]

daytime_features_in_daily = [
    col for col in daytime_only_features
    if col in daily_feature_source.columns
]

missing_daytime_features_in_daily = sorted(
    set(daytime_only_features) - set(daytime_features_in_daily)
)

print("=== Episode Index Summary ===")
print("episodes:", len(episode_index))
print("participants:", episode_index[ID_COL].nunique())
print("unique sleep_episode_id:", episode_index["sleep_episode_id"].nunique())
print("target mean:", episode_index[TARGET_COL].mean())
print("sleep_start_date range:", episode_index["sleep_start_date"].min(), "to", episode_index["sleep_start_date"].max())
print("sleep_end_date range:", episode_index["sleep_end_date"].min(), "to", episode_index["sleep_end_date"].max())

print("\n=== Daily Feature Source Summary ===")
print("daily_feature_source rows:", len(daily_feature_source))
print("daily_feature_source participants:", daily_feature_source[ID_COL].nunique())
print("available non-sleep daily feature columns:", len(available_daily_feature_columns))
print("daytime_only features requested:", len(daytime_only_features))
print("daytime_only features found in daily source:", len(daytime_features_in_daily))
print("missing daytime_only features in daily source:", len(missing_daytime_features_in_daily))
if missing_daytime_features_in_daily:
    display(pd.DataFrame({"missing_feature": missing_daytime_features_in_daily}))

# start_date 당일 daily feature join coverage
daily_key = daily_feature_source[[ID_COL, "calendar_date"] + daytime_features_in_daily].copy()
daily_key = daily_key.rename(columns={"calendar_date": "feature_date"})

same_start_join = episode_index.merge(
    daily_key,
    left_on=[ID_COL, "sleep_start_date"],
    right_on=[ID_COL, "feature_date"],
    how="left",
    indicator="same_start_feature_join",
)

same_start_join["same_start_feature_available"] = (
    same_start_join["same_start_feature_join"] == "both"
)

# start_date - 1 day previous feature join coverage
episode_index["sleep_start_previous_date"] = (
    episode_index["sleep_start_date"] - pd.Timedelta(days=1)
)

prev_start_join = episode_index.merge(
    daily_key,
    left_on=[ID_COL, "sleep_start_previous_date"],
    right_on=[ID_COL, "feature_date"],
    how="left",
    indicator="previous_start_feature_join",
)

prev_start_join["previous_start_feature_available"] = (
    prev_start_join["previous_start_feature_join"] == "both"
)

coverage_summary = pd.DataFrame(
    [
        {
            "feature_alignment": "sleep_start_date daily features",
            "matched_rows": int(same_start_join["same_start_feature_available"].sum()),
            "total_rows": len(same_start_join),
            "coverage": float(same_start_join["same_start_feature_available"].mean()),
        },
        {
            "feature_alignment": "sleep_start_date - 1 daily features",
            "matched_rows": int(prev_start_join["previous_start_feature_available"].sum()),
            "total_rows": len(prev_start_join),
            "coverage": float(prev_start_join["previous_start_feature_available"].mean()),
        },
    ]
)

print("\n=== Start-Date Feature Join Coverage ===")
display(coverage_summary)

# daytime_only feature missingness after start-date join
if daytime_features_in_daily:
    same_start_daytime_missing = (
        same_start_join[daytime_features_in_daily]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_rate_after_start_date_join")
        .reset_index()
        .rename(columns={"index": "feature"})
    )

    print("\n=== Daytime Feature Missingness After sleep_start_date Join ===")
    display(same_start_daytime_missing)

# 기존 previous_day dataset이 sleep_start_date 관점에서 무엇을 의미했는지 확인합니다.
previous_alignment = episode_index[
    [ID_COL, "calendar_date", "sleep_start_date", "sleep_end_date", "startTime", "endTime", TARGET_COL]
].merge(
    previous_day_df[[ID_COL, "calendar_date", "feature_date", "feature_lag_days"]],
    on=[ID_COL, "calendar_date"],
    how="left",
)

previous_alignment["previous_feature_matches_sleep_start_date"] = (
    previous_alignment["feature_date"].dt.normalize()
    == previous_alignment["sleep_start_date"].dt.normalize()
)

previous_alignment["previous_feature_is_before_sleep_start_date"] = (
    previous_alignment["feature_date"].dt.normalize()
    < previous_alignment["sleep_start_date"].dt.normalize()
)

previous_alignment["previous_feature_lag_from_sleep_start_days"] = (
    previous_alignment["sleep_start_date"].dt.normalize()
    - previous_alignment["feature_date"].dt.normalize()
).dt.days

previous_alignment_summary = pd.DataFrame(
    {
        "metric": [
            "episodes",
            "previous_day_rows_matched",
            "feature_matches_sleep_start_date_rows",
            "feature_before_sleep_start_date_rows",
            "feature_matches_sleep_start_date_ratio_among_matched",
            "feature_before_sleep_start_date_ratio_among_matched",
        ],
        "value": [
            len(previous_alignment),
            int(previous_alignment["feature_date"].notna().sum()),
            int(previous_alignment["previous_feature_matches_sleep_start_date"].sum()),
            int(previous_alignment["previous_feature_is_before_sleep_start_date"].sum()),
            float(
                previous_alignment.loc[
                    previous_alignment["feature_date"].notna(),
                    "previous_feature_matches_sleep_start_date",
                ].mean()
            ),
            float(
                previous_alignment.loc[
                    previous_alignment["feature_date"].notna(),
                    "previous_feature_is_before_sleep_start_date",
                ].mean()
            ),
        ],
    }
)

print("\n=== Existing Previous-Day Alignment Relative To sleep_start_date ===")
display(previous_alignment_summary)

print("\nprevious_feature_lag_from_sleep_start_days distribution:")
display(
    previous_alignment["previous_feature_lag_from_sleep_start_days"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("rows")
    .reset_index()
    .rename(columns={"index": "lag_days_from_sleep_start"})
)

print("\n=== Example Episode Alignment Rows ===")
display(
    previous_alignment[
        [
            ID_COL,
            "calendar_date",
            "sleep_start_date",
            "sleep_end_date",
            "startTime",
            "endTime",
            "feature_date",
            "previous_feature_lag_from_sleep_start_days",
            "previous_feature_matches_sleep_start_date",
            "previous_feature_is_before_sleep_start_date",
            TARGET_COL,
        ]
    ].head(20)
)

print("\n=== Preliminary Feature Design Decision ===")
print("Design A: use sleep_start_date - 1 daily features as conservative baseline.")
print("Design B: use sleep_start_date daily features as same-day pre-sleep approximation.")
print("Design C: use intraday cutoff before startTime if raw intraday source is available.")

=== Episode Index Summary ===
episodes: 3551
participants: 69
unique sleep_episode_id: 3551
target mean: 0.39369191776964235
sleep_start_date range: 2021-05-24 00:00:00 to 2022-01-21 00:00:00
sleep_end_date range: 2021-05-24 00:00:00 to 2022-01-22 00:00:00

=== Daily Feature Source Summary ===
daily_feature_source rows: 3551
daily_feature_source participants: 69
available non-sleep daily feature columns: 113
daytime_only features requested: 19
daytime_only features found in daily source: 11
missing daytime_only features in daily source: 8


,missing_feature
0,calories_record_count_missing_ind
1,calories_sum_missing_ind
2,lightly_active_minutes_sum_missing_ind
3,moderately_active_minutes_sum_missing_ind
4,sedentary_minutes_sum_missing_ind
5,steps_record_count_missing_ind
6,steps_sum_missing_ind
7,very_active_minutes_sum_missing_ind



=== Start-Date Feature Join Coverage ===


,feature_alignment,matched_rows,total_rows,coverage
0,sleep_start_date daily features,3398,3551,0.956914
1,sleep_start_date - 1 daily features,3101,3551,0.873275



=== Daytime Feature Missingness After sleep_start_date Join ===


...


previous_feature_lag_from_sleep_start_days distribution:


,previous_feature_lag_from_sleep_start_days,rows
0,0.0,1196
1,1.0,1920
2,NaN,435



=== Example Episode Alignment Rows ===


,participant_object_id,calendar_date,sleep_start_date,sleep_end_date,startTime,endTime,feature_date,previous_feature_lag_from_sleep_start_days,previous_feature_matches_sleep_start_date,previous_feature_is_before_sleep_start_date,good_sleep_label
0,621e2e8e67b776a24055b564,2021-05-24,2021-05-24,2021-05-24,2021-05-24 00:40:00,2021-05-24 09:21:00,NaT,NaN,False,False,1
1,621e2e8e67b776a24055b564,2021-05-25,2021-05-24,2021-05-25,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,0.0,True,False,1
2,621e2e8e67b776a24055b564,2021-05-26,2021-05-25,2021-05-26,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,0.0,True,False,1
3,621e2e8e67b776a24055b564,2021-05-27,2021-05-26,2021-05-27,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,0.0,True,False,1
4,621e2e8e67b776a24055b564,2021-05-28,2021-05-27,2021-05-28,2021-05-27 23:37:00,2021-05-28 08:58:30,2021-05-27,0.0,True,False,1
5,621e2e8e67b776a24055b564,2021-05-29,2021-05-28,2021-05-29,2021-05-28 23:50:00,2021-05-29 08:48:00,2021-05-28,0.0,True,False,1
6,621e2e8e67b776a24055b564,2021-05-30,2021-05-30,2021-05-30,2021-05-30 00:30:00,2021-05-30 09:45:30,2021-05-29,1.0,False,True,1
7,621e2e8e67b776a24055b564,2021-05-31,2021-05-30,2021-05-31,2021-05-30 23:29:30,2021-05-31 09:08:30,2021-05-30,0.0,True,False,1
8,621e2e8e67b776a24055b564,2021-06-01,2021-05-31,2021-06-01,2021-05-31 23:42:30,2021-06-01 09:26:30,2021-05-31,0.0,True,False,1
9,621e2e8e67b776a24055b564,2021-06-02,2021-06-01,2021-06-02,2021-06-01 23:58:30,2021-06-02 09:32:30,2021-06-01,0.0,True,False,1



=== Preliminary Feature Design Decision ===
Design A: use sleep_start_date - 1 daily features as conservative baseline.
Design B: use sleep_start_date daily features as same-day pre-sleep approximation.
Design C: use intraday cutoff before startTime if raw intraday source is available.


In [4]:
# Cell 4. Design B feature policy 초안 생성: sleep_start_date daily feature approximation
# 원하는 결과:
# - modeling_dataset_daily의 non-sleep feature를 feature family별로 분류한다.
# - pre-sleep 목적에서 include / maybe / exclude를 초안으로 나눈다.
# - 특히 sleep-derived leakage 위험 feature를 제외한다.
# - Design B에서 사용할 1차 candidate feature 목록을 만든다.
# - 아직 파일 저장은 하지 않는다.

ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
TARGET_COL = "good_sleep_label"

# modeling_dataset_daily 기준 sleep outcome/id 컬럼
sleep_outcome_or_id_columns = [
    ID_COL,
    DATE_COL,
    "startTime",
    "endTime",
    "minutesAsleep",
    "minutesAwake",
    "timeInBed",
    "sleep_duration_hours",
    "time_in_bed_hours",
    "awake_ratio",
    "wake_minutes",
    "wake_ratio",
    "asleep_minutes",
    "awake_minutes",
    "classic_asleep_ratio",
    "classic_awake_ratio",
    TARGET_COL,
]

candidate_daily_columns = [
    col for col in modeling_daily.columns
    if col not in sleep_outcome_or_id_columns
]

def infer_feature_family(feature_name):
    name = feature_name.lower()

    if name.startswith("stress_"):
        return "stress"
    if name.startswith("hrv_summary_") or name.startswith("hrv_detail_"):
        return "hrv"
    if name.startswith("respiratory_"):
        return "sleep_respiratory"
    if name.startswith("spo2_"):
        return "sleep_spo2"
    if name.startswith("temperature_") or name.startswith("wrist_temperature_"):
        return "temperature"
    if name.startswith("resting_hr_"):
        return "resting_hr"
    if (
        "steps" in name
        or "calories" in name
        or "active_minutes" in name
        or "sedentary_minutes" in name
        or "activity" in name
    ):
        return "daytime_activity"
    if (
        name.startswith("mood_")
        or name.startswith("place_")
        or name.startswith("survey_")
    ):
        return "survey_context"
    return "other"

def infer_pre_sleep_policy(feature_name, feature_family):
    name = feature_name.lower()

    # target sleep episode에서 산출된 수면 생리/호흡/SpO2/HRV 계열은 pre-sleep feature로 금지한다.
    if feature_family in ["sleep_respiratory", "sleep_spo2", "hrv"]:
        return "exclude_target_sleep_or_post_sleep_risk"

    # stress_sleep_points는 이름상 sleep 관련 점수라 제외한다.
    if "sleep" in name and feature_family == "stress":
        return "exclude_sleep_related_stress_component"

    # sleep_start_date의 daily aggregate는 bedtime 이후 일부 데이터가 섞일 수 있어 maybe로 둔다.
    if feature_family in ["daytime_activity", "resting_hr", "temperature", "stress", "survey_context"]:
        return "maybe_include_daily_start_date_approximation"

    return "review_manually"

feature_policy_rows = []
for feature in candidate_daily_columns:
    family = infer_feature_family(feature)
    policy = infer_pre_sleep_policy(feature, family)

    feature_policy_rows.append(
        {
            "feature": feature,
            "feature_family": family,
            "pre_sleep_policy": policy,
            "is_daytime_only_feature": feature in daytime_only_features,
        }
    )

feature_policy = pd.DataFrame(feature_policy_rows)

print("=== Design B Feature Policy Summary ===")
display(
    feature_policy
    .groupby(["feature_family", "pre_sleep_policy"])
    .agg(
        features=("feature", "count"),
        daytime_only_features=("is_daytime_only_feature", "sum"),
    )
    .reset_index()
    .sort_values(["pre_sleep_policy", "feature_family"])
)

print("\n=== Feature Policy Detail ===")
display(feature_policy.sort_values(["pre_sleep_policy", "feature_family", "feature"]).reset_index(drop=True))

# 1차 Design B 후보: maybe include 계열만 사용.
# 단, 현재 목적의 첫 모델은 conservative하게 daytime activity/resting_hr 중심으로 시작하는 것이 좋다.
design_b_broad_candidate_features = feature_policy[
    feature_policy["pre_sleep_policy"].eq("maybe_include_daily_start_date_approximation")
]["feature"].tolist()

design_b_daytime_only_available_features = [
    feature for feature in daytime_only_features
    if feature in design_b_broad_candidate_features
]

# missing indicator는 modeling_dataset_daily 원본에는 일부 없으므로, 현재 존재하는 11개만 우선 사용한다.
print("\n=== Design B Candidate Feature Counts ===")
print("candidate_daily_columns:", len(candidate_daily_columns))
print("design_b_broad_candidate_features:", len(design_b_broad_candidate_features))
print("design_b_daytime_only_available_features:", len(design_b_daytime_only_available_features))
print("daytime_only requested:", len(daytime_only_features))
print("daytime_only available in Design B:", design_b_daytime_only_available_features)

# sleep_start_date daily features를 붙인 Design B preview를 만든다.
daily_feature_source_for_design_b = modeling_daily.copy()
daily_feature_source_for_design_b[DATE_COL] = pd.to_datetime(
    daily_feature_source_for_design_b[DATE_COL],
    errors="coerce",
).dt.normalize()

design_b_episode_features = episode_index.merge(
    daily_feature_source_for_design_b[
        [ID_COL, DATE_COL] + design_b_daytime_only_available_features
    ].rename(columns={DATE_COL: "feature_date"}),
    left_on=[ID_COL, "sleep_start_date"],
    right_on=[ID_COL, "feature_date"],
    how="left",
    indicator="design_b_join",
)

design_b_episode_features["design_b_feature_available"] = (
    design_b_episode_features["design_b_join"] == "both"
)

print("\n=== Design B Daytime-Only Approximation Coverage ===")
design_b_coverage = pd.DataFrame(
    {
        "metric": [
            "episodes",
            "matched_rows",
            "coverage",
            "feature_count",
            "target_mean_all",
            "target_mean_matched",
        ],
        "value": [
            len(design_b_episode_features),
            int(design_b_episode_features["design_b_feature_available"].sum()),
            float(design_b_episode_features["design_b_feature_available"].mean()),
            len(design_b_daytime_only_available_features),
            float(design_b_episode_features[TARGET_COL].mean()),
            float(
                design_b_episode_features.loc[
                    design_b_episode_features["design_b_feature_available"],
                    TARGET_COL,
                ].mean()
            ),
        ],
    }
)
display(design_b_coverage)

print("\n=== Design B Feature Missingness Among Matched Rows ===")
matched_design_b = design_b_episode_features[
    design_b_episode_features["design_b_feature_available"]
].copy()

if design_b_daytime_only_available_features:
    design_b_missing = (
        matched_design_b[design_b_daytime_only_available_features]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_rate_among_matched")
        .reset_index()
        .rename(columns={"index": "feature"})
    )
    display(design_b_missing)

print("\n=== Design B Example Rows ===")
display(
    design_b_episode_features[
        [
            ID_COL,
            "sleep_episode_id",
            "startTime",
            "endTime",
            "sleep_start_date",
            "sleep_end_date",
            "feature_date",
            "design_b_feature_available",
            TARGET_COL,
        ] + design_b_daytime_only_available_features
    ].head(20)
)

print("\n=== Design B Interpretation ===")
print("Design B uses sleep_start_date daily aggregate features as a pre-sleep same-day approximation.")
print("This matches the target purpose better than sleep_end_date same-date features, but daily aggregates may include small post-bedtime activity unless intraday cutoff is available.")
print("Next step: check whether raw/intraday sources exist for strict Design C.")

=== Design B Feature Policy Summary ===


,feature_family,pre_sleep_policy,features,daytime_only_features
6,stress,exclude_sleep_related_stress_component,1,0
1,hrv,exclude_target_sleep_or_post_sleep_risk,21,0
4,sleep_respiratory,exclude_target_sleep_or_post_sleep_risk,7,0
5,sleep_spo2,exclude_target_sleep_or_post_sleep_risk,4,0
0,daytime_activity,maybe_include_daily_start_date_approximation,8,8
3,resting_hr,maybe_include_daily_start_date_approximation,3,3
7,stress,maybe_include_daily_start_date_approximation,6,0
8,survey_context,maybe_include_daily_start_date_approximation,47,0
9,temperature,maybe_include_daily_start_date_approximation,4,0
2,other,review_manually,12,0



=== Feature Policy Detail ===


,feature,feature_family,pre_sleep_policy,is_daytime_only_feature
0,stress_sleep_points_mean,stress,exclude_sleep_related_stress_component,False
1,hrv_detail_coverage_max,hrv,exclude_target_sleep_or_post_sleep_risk,False
2,hrv_detail_coverage_mean,hrv,exclude_target_sleep_or_post_sleep_risk,False
3,hrv_detail_coverage_min,hrv,exclude_target_sleep_or_post_sleep_risk,False
4,hrv_detail_coverage_std,hrv,exclude_target_sleep_or_post_sleep_risk,False
5,hrv_detail_high_frequency_max,hrv,exclude_target_sleep_or_post_sleep_risk,False
6,hrv_detail_high_frequency_mean,hrv,exclude_target_sleep_or_post_sleep_risk,False
7,hrv_detail_high_frequency_min,hrv,exclude_target_sleep_or_post_sleep_risk,False
8,hrv_detail_high_frequency_std,hrv,exclude_target_sleep_or_post_sleep_risk,False
9,hrv_detail_low_frequency_max,hrv,exclude_target_sleep_or_post_sleep_risk,False



=== Design B Candidate Feature Counts ===
candidate_daily_columns: 113
design_b_broad_candidate_features: 68
design_b_daytime_only_available_features: 11
daytime_only requested: 19
daytime_only available in Design B: ['resting_hr_resting_heart_rate_mean', 'resting_hr_error_mean', 'resting_hr_record_count', 'lightly_active_minutes_sum', 'moderately_active_minutes_sum', 'sedentary_minutes_sum', 'very_active_minutes_sum', 'steps_sum', 'steps_record_count', 'calories_sum', 'calories_record_count']

=== Design B Daytime-Only Approximation Coverage ===


...


=== Design B Feature Missingness Among Matched Rows ===


,feature,missing_rate_among_matched
0,steps_record_count,0.001766
1,steps_sum,0.001766
2,lightly_active_minutes_sum,0.000589
3,very_active_minutes_sum,0.000589
4,sedentary_minutes_sum,0.000589
5,calories_sum,0.000589
6,moderately_active_minutes_sum,0.000589
7,calories_record_count,0.000589
8,resting_hr_record_count,0.000000
9,resting_hr_resting_heart_rate_mean,0.000000



=== Design B Example Rows ===


,participant_object_id,sleep_episode_id,startTime,endTime,sleep_start_date,sleep_end_date,feature_date,design_b_feature_available,good_sleep_label,resting_hr_resting_heart_rate_mean,resting_hr_error_mean,resting_hr_record_count,lightly_active_minutes_sum,moderately_active_minutes_sum,sedentary_minutes_sum,very_active_minutes_sum,steps_sum,steps_record_count,calories_sum,calories_record_count
0,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210524004000__20210524092100,2021-05-24 00:40:00,2021-05-24 09:21:00,2021-05-24,2021-05-24,2021-05-24,True,1,62.073070,6.787090,1.0,149.0,24.0,713.0,33.0,8833.0,685.0,2351.59,1440.0
1,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210524234830__20210525085630,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,2021-05-25,2021-05-24,True,1,62.073070,6.787090,1.0,149.0,24.0,713.0,33.0,8833.0,685.0,2351.59,1440.0
2,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210525234630__20210526090630,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,2021-05-26,2021-05-25,True,1,62.121476,6.787088,1.0,132.0,25.0,704.0,31.0,9727.0,674.0,2332.08,1440.0
3,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210526232130__20210527094830,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,2021-05-27,2021-05-26,True,1,62.263999,6.787087,1.0,112.0,27.0,710.0,31.0,8253.0,577.0,2262.30,1440.0
4,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210527233700__20210528085830,2021-05-27 23:37:00,2021-05-28 08:58:30,2021-05-27,2021-05-28,2021-05-27,True,1,62.368900,6.787087,1.0,133.0,21.0,622.0,37.0,9015.0,614.0,2325.10,1440.0
5,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210528235000__20210529084800,2021-05-28 23:50:00,2021-05-29 08:48:00,2021-05-28,2021-05-29,2021-05-28,True,1,61.965409,6.787087,1.0,136.0,42.0,647.0,54.0,12949.0,761.0,2586.76,1440.0
6,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210530003000__20210530094530,2021-05-30 00:30:00,2021-05-30 09:45:30,2021-05-30,2021-05-30,2021-05-30,True,1,63.359720,6.787087,1.0,113.0,9.0,763.0,0.0,3796.0,403.0,1968.24,1440.0
7,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210530232930__20210531090830,2021-05-30 23:29:30,2021-05-31 09:08:30,2021-05-30,2021-05-31,2021-05-30,True,1,63.359720,6.787087,1.0,113.0,9.0,763.0,0.0,3796.0,403.0,1968.24,1440.0
8,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210531234230__20210601092630,2021-05-31 23:42:30,2021-06-01 09:26:30,2021-05-31,2021-06-01,2021-05-31,True,1,63.121265,6.787087,1.0,149.0,23.0,655.0,34.0,9245.0,569.0,2300.02,1440.0
9,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210601235830__20210602093230,2021-06-01 23:58:30,2021-06-02 09:32:30,2021-06-01,2021-06-02,2021-06-01,True,1,62.419622,6.787087,1.0,106.0,30.0,691.0,29.0,8422.0,600.0,2227.17,1440.0



=== Design B Interpretation ===
Design B uses sleep_start_date daily aggregate features as a pre-sleep same-day approximation.
This matches the target purpose better than sleep_end_date same-date features, but daily aggregates may include small post-bedtime activity unless intraday cutoff is available.
Next step: check whether raw/intraday sources exist for strict Design C.


In [5]:
# Cell 5. Design C 가능성 확인: raw/intraday timestamp feature source 탐색
# 원하는 결과:
# - bedtime 전까지만 자를 수 있는 intraday/time-stamped wearable source가 있는지 찾는다.
# - steps/calories/activity minutes 관련 raw 또는 processed 파일을 넓게 스캔한다.
# - MongoDB/BSON source를 봐야 하는지 판단한다.
# - 아직 파일 저장은 하지 않는다.

RAW_DIR = DATA_DIR / "raw"

search_roots = [
    DATA_DIR,
    PROCESSED_DIR,
    RAW_DIR,
]

intraday_keywords = [
    "intraday",
    "minute",
    "minutes",
    "steps",
    "calories",
    "activity",
    "heart",
    "hr",
    "fitbit",
]

candidate_extensions = [".csv", ".json", ".jsonl", ".parquet", ".pkl", ".joblib", ".bson"]

file_scan_rows = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        path_lower = str(path).lower()
        suffix = path.suffix.lower()

        if suffix not in candidate_extensions:
            continue

        if any(keyword in path_lower for keyword in intraday_keywords):
            file_scan_rows.append(
                {
                    "relative_path": str(path.relative_to(PROJECT_ROOT)),
                    "suffix": suffix,
                    "size_mb": path.stat().st_size / (1024 ** 2),
                    "name": path.name,
                }
            )

intraday_file_candidates = (
    pd.DataFrame(file_scan_rows)
    .sort_values(["suffix", "relative_path"])
    .reset_index(drop=True)
)

print("=== Intraday / Time-Stamped File Candidates ===")
display(intraday_file_candidates)

# CSV 후보는 컬럼과 샘플을 조금 더 확인합니다.
csv_candidates = []
if not intraday_file_candidates.empty:
    csv_candidates = [
        PROJECT_ROOT / rel
        for rel in intraday_file_candidates.loc[
            intraday_file_candidates["suffix"].eq(".csv"),
            "relative_path",
        ]
    ]

csv_preview_rows = []
csv_preview_tables = {}

for path in csv_candidates:
    try:
        preview = pd.read_csv(path, encoding="utf-8-sig", nrows=10)
        columns = list(preview.columns)

        time_like_columns = [
            col for col in columns
            if any(token in col.lower() for token in ["time", "date", "datetime", "timestamp", "minute"])
        ]

        value_like_columns = [
            col for col in columns
            if any(token in col.lower() for token in ["steps", "calories", "active", "sedentary", "heart", "hr"])
        ]

        csv_preview_rows.append(
            {
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "rows_previewed": len(preview),
                "columns_count": len(columns),
                "time_like_columns": time_like_columns,
                "value_like_columns": value_like_columns,
                "columns": columns,
            }
        )

        # 너무 많은 preview는 피하고, time/value 컬럼이 있으면 저장해둡니다.
        if time_like_columns or value_like_columns:
            csv_preview_tables[str(path.relative_to(PROJECT_ROOT))] = preview.head(5)

    except Exception as exc:
        csv_preview_rows.append(
            {
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "rows_previewed": None,
                "columns_count": None,
                "time_like_columns": f"READ_ERROR: {exc}",
                "value_like_columns": None,
                "columns": None,
            }
        )

csv_preview_summary = pd.DataFrame(csv_preview_rows)

print("\n=== CSV Candidate Column Summary ===")
display(csv_preview_summary)

# 핵심 processed daily aggregate 파일의 record_count가 하루 전체인지 확인합니다.
daily_aggregate_paths = [
    PROCESSED_DIR / "daily_aggregates" / "fitbit_steps_daily.csv",
    PROCESSED_DIR / "daily_aggregates" / "fitbit_calories_daily.csv",
    PROCESSED_DIR / "daily_aggregates" / "fitbit_activity_minutes_daily.csv",
    PROCESSED_DIR / "daily_aggregates" / "fitbit_resting_heart_rate_daily.csv",
]

print("\n=== Daily Aggregate Preview ===")
for path in daily_aggregate_paths:
    if path.exists():
        preview = pd.read_csv(path, encoding="utf-8-sig", nrows=5)
        print(f"\n{path.relative_to(PROJECT_ROOT)}")
        display(preview)

print("\n=== Selected CSV Previews With Time/Value Columns ===")
for rel_path, preview in list(csv_preview_tables.items())[:20]:
    print(f"\n{rel_path}")
    display(preview)

print("\n=== Design C Preliminary Decision Guide ===")
print("If intraday steps/calories/activity files with timestamp columns exist, Design C is feasible.")
print("If only daily aggregate files exist, Design C requires returning to MongoDB/BSON source.")
print("If neither intraday nor source timestamp data is accessible, Design B is the practical first model.")

=== Intraday / Time-Stamped File Candidates ===


,relative_path,suffix,size_mb,name
0,data\processed\daily_aggregates\fitbit_activity_minutes_daily.csv,.csv,0.334445,fitbit_activity_minutes_daily.csv
1,data\processed\daily_aggregates\fitbit_activity_minutes_daily.csv,.csv,0.334445,fitbit_activity_minutes_daily.csv
2,data\processed\daily_aggregates\fitbit_calories_daily.csv,.csv,0.326138,fitbit_calories_daily.csv
3,data\processed\daily_aggregates\fitbit_calories_daily.csv,.csv,0.326138,fitbit_calories_daily.csv
4,data\processed\daily_aggregates\fitbit_daily_hrv_summary_daily.csv,.csv,0.138433,fitbit_daily_hrv_summary_daily.csv
5,data\processed\daily_aggregates\fitbit_daily_hrv_summary_daily.csv,.csv,0.138433,fitbit_daily_hrv_summary_daily.csv
6,data\processed\daily_aggregates\fitbit_daily_spo2_daily.csv,.csv,0.065606,fitbit_daily_spo2_daily.csv
7,data\processed\daily_aggregates\fitbit_daily_spo2_daily.csv,.csv,0.065606,fitbit_daily_spo2_daily.csv
8,data\processed\daily_aggregates\fitbit_hrv_details_daily.csv,.csv,0.602516,fitbit_hrv_details_daily.csv
9,data\processed\daily_aggregates\fitbit_hrv_details_daily.csv,.csv,0.602516,fitbit_hrv_details_daily.csv



=== CSV Candidate Column Summary ===


,relative_path,rows_previewed,columns_count,time_like_columns,value_like_columns,columns
0,data\processed\daily_aggregates\fitbit_activity_minutes_daily.csv,10,6,"[calendar_date, lightly_active_minutes_sum, moderately_active_minutes_sum, sedentary_minutes_sum, very_active_minutes_sum]","[lightly_active_minutes_sum, moderately_active_minutes_sum, sedentary_minutes_sum, very_active_minutes_sum]","[participant_object_id, calendar_date, lightly_active_minutes_sum, moderately_active_minutes_sum, sedentary_minutes_sum, very_active_minutes_sum]"
1,data\processed\daily_aggregates\fitbit_activity_minutes_daily.csv,10,6,"[calendar_date, lightly_active_minutes_sum, moderately_active_minutes_sum, sedentary_minutes_sum, very_active_minutes_sum]","[lightly_active_minutes_sum, moderately_active_minutes_sum, sedentary_minutes_sum, very_active_minutes_sum]","[participant_object_id, calendar_date, lightly_active_minutes_sum, moderately_active_minutes_sum, sedentary_minutes_sum, very_active_minutes_sum]"
2,data\processed\daily_aggregates\fitbit_calories_daily.csv,10,4,[calendar_date],"[calories_sum, calories_record_count]","[participant_object_id, calendar_date, calories_sum, calories_record_count]"
3,data\processed\daily_aggregates\fitbit_calories_daily.csv,10,4,[calendar_date],"[calories_sum, calories_record_count]","[participant_object_id, calendar_date, calories_sum, calories_record_count]"
4,data\processed\daily_aggregates\fitbit_daily_hrv_summary_daily.csv,10,6,[calendar_date],"[hrv_summary_rmssd_mean, hrv_summary_nremhr_mean, hrv_summary_entropy_mean, hrv_summary_record_count]","[participant_object_id, calendar_date, hrv_summary_rmssd_mean, hrv_summary_nremhr_mean, hrv_summary_entropy_mean, hrv_summary_record_count]"
5,data\processed\daily_aggregates\fitbit_daily_hrv_summary_daily.csv,10,6,[calendar_date],"[hrv_summary_rmssd_mean, hrv_summary_nremhr_mean, hrv_summary_entropy_mean, hrv_summary_record_count]","[participant_object_id, calendar_date, hrv_summary_rmssd_mean, hrv_summary_nremhr_mean, hrv_summary_entropy_mean, hrv_summary_record_count]"
6,data\processed\daily_aggregates\fitbit_daily_spo2_daily.csv,10,6,[calendar_date],[],"[participant_object_id, calendar_date, spo2_average_value_mean, spo2_lower_bound_mean, spo2_upper_bound_mean, spo2_record_count]"
7,data\processed\daily_aggregates\fitbit_daily_spo2_daily.csv,10,6,[calendar_date],[],"[participant_object_id, calendar_date, spo2_average_value_mean, spo2_lower_bound_mean, spo2_upper_bound_mean, spo2_record_count]"
8,data\processed\daily_aggregates\fitbit_hrv_details_daily.csv,10,19,[calendar_date],"[hrv_detail_rmssd_mean, hrv_detail_rmssd_std, hrv_detail_rmssd_min, hrv_detail_rmssd_max, hrv_detail_coverage_mean, hrv_detail_coverage_std, hrv_detail_cove...","[participant_object_id, calendar_date, hrv_detail_rmssd_mean, hrv_detail_rmssd_std, hrv_detail_rmssd_min, hrv_detail_rmssd_max, hrv_detail_coverage_mean, hr..."
9,data\processed\daily_aggregates\fitbit_hrv_details_daily.csv,10,19,[calendar_date],"[hrv_detail_rmssd_mean, hrv_detail_rmssd_std, hrv_detail_rmssd_min, hrv_detail_rmssd_max, hrv_detail_coverage_mean, hrv_detail_coverage_std, hrv_detail_cove...","[participant_object_id, calendar_date, hrv_detail_rmssd_mean, hrv_detail_rmssd_std, hrv_detail_rmssd_min, hrv_detail_rmssd_max, hrv_detail_coverage_mean, hr..."



=== Daily Aggregate Preview ===

data\processed\daily_aggregates\fitbit_steps_daily.csv


...


data\raw\extracted_variables\fitbit_respiratory_rate_summary.csv


,mongo_doc_id,participant_object_id,fitbit_type,timestamp,full_sleep_breathing_rate,full_sleep_signal_to_noise,full_sleep_standard_deviation,deep_sleep_breathing_rate,light_sleep_breathing_rate,rem_sleep_breathing_rate
0,62cc2021b41dcd4b1bfe9344,621e2e8e67b776a24055b564,Respiratory Rate Summary,2021-05-30T09:45:30,15.2,7.606,0.9,13.8,NaN,15.2
1,62cc2021b41dcd4b1bfe9350,621e2e8e67b776a24055b564,Respiratory Rate Summary,2021-06-06T10:22:00,14.4,10.089,0.8,14.8,NaN,17.0
2,62cc2021b41dcd4b1bfe935c,621e2e8e67b776a24055b564,Respiratory Rate Summary,2021-06-24T09:22:00,14.8,11.342,1.2,15.0,NaN,10.8
3,62cc2021b41dcd4b1bfe9368,621e2e8e67b776a24055b564,Respiratory Rate Summary,2021-07-07T09:44:30,16.2,6.160,1.5,15.0,NaN,19.0
4,62cc2021b41dcd4b1bfe9374,621e2e8e67b776a24055b564,Respiratory Rate Summary,2021-07-19T10:23:30,16.2,7.237,1.1,15.8,NaN,19.2



data\raw\extracted_variables\fitbit_resting_heart_rate.csv


,mongo_doc_id,participant_object_id,fitbit_type,date_time,value_date,resting_heart_rate,error
0,62cc2021b41dcd4b1bfd9bde,621e2e8e67b776a24055b564,resting_heart_rate,05/25/21 00:00:00,05/25/21,62.121476,6.787088
1,62cc2021b41dcd4b1bfd9bee,621e2e8e67b776a24055b564,resting_heart_rate,06/10/21 00:00:00,06/10/21,59.877669,10.699845
2,62cc2021b41dcd4b1bfd9bfa,621e2e8e67b776a24055b564,resting_heart_rate,06/22/21 00:00:00,06/22/21,57.934354,6.790485
3,62cc2021b41dcd4b1bfd9c06,621e2e8e67b776a24055b564,resting_heart_rate,07/04/21 00:00:00,07/04/21,57.861243,11.628634
4,62cc2021b41dcd4b1bfd9c12,621e2e8e67b776a24055b564,resting_heart_rate,07/16/21 00:00:00,07/16/21,60.603417,6.787094



=== Design C Preliminary Decision Guide ===
If intraday steps/calories/activity files with timestamp columns exist, Design C is feasible.
If only daily aggregate files exist, Design C requires returning to MongoDB/BSON source.
If neither intraday nor source timestamp data is accessible, Design B is the practical first model.


In [6]:
# Cell 6. MongoDB/raw source에서 strict intraday Design C 가능성 확인
# 원하는 결과:
# - MongoDB가 사용 가능하면 fitbit collection의 type 목록을 샘플/집계로 확인한다.
# - intraday steps/calories/activity timestamp document가 있는지 찾는다.
# - MongoDB가 불가능하면 raw 폴더의 BSON/JSON 후보를 확인한다.
# - 전체 fitbit collection을 pandas로 로드하지 않는다.

from pathlib import Path
import subprocess
import sys

MONGO_URI = "mongodb://localhost:27017"
MONGO_DB = "rais_anonymized"
MONGO_COLLECTION = "fitbit"

print("=== MongoDB Intraday Source Check ===")

mongo_available = False
mongo_error = None

try:
    from pymongo import MongoClient

    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    client.admin.command("ping")
    mongo_available = True
    print("MongoDB available:", mongo_available)
except Exception as exc:
    mongo_error = exc
    print("MongoDB available:", mongo_available)
    print("MongoDB error:", repr(exc))

if mongo_available:
    db = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]

    print("\n=== Collection Basic Counts ===")
    try:
        total_docs = collection.estimated_document_count()
        print("estimated fitbit documents:", total_docs)
    except Exception as exc:
        print("estimated count error:", repr(exc))

    print("\n=== Fitbit Type Counts Containing Activity/Step/Calorie/Intraday Keywords ===")
    keyword_regex = {
        "$regex": "step|calorie|activ|minute|intraday|heart",
        "$options": "i",
    }

    pipeline = [
        {"$match": {"type": keyword_regex}},
        {"$group": {"_id": "$type", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}},
        {"$limit": 100},
    ]

    type_counts = list(collection.aggregate(pipeline, allowDiskUse=True))
    type_counts_df = pd.DataFrame(
        [
            {"fitbit_type": row["_id"], "documents": row["count"]}
            for row in type_counts
        ]
    )

    display(type_counts_df)

    print("\n=== Sample Documents For Candidate Types ===")
    candidate_types = []
    if not type_counts_df.empty:
        candidate_types = type_counts_df["fitbit_type"].tolist()

    sample_rows = []
    for fitbit_type in candidate_types[:30]:
        doc = collection.find_one({"type": fitbit_type})
        if doc is None:
            continue

        top_level_keys = sorted(list(doc.keys()))
        value = doc.get("value", None)

        if isinstance(value, dict):
            value_type = "dict"
            value_keys = sorted(list(value.keys()))[:30]
            value_sample = str({k: value.get(k) for k in value_keys[:5]})[:500]
        elif isinstance(value, list):
            value_type = "list"
            value_keys = []
            value_sample = str(value[:2])[:500]
        else:
            value_type = type(value).__name__
            value_keys = []
            value_sample = str(value)[:500]

        sample_rows.append(
            {
                "fitbit_type": fitbit_type,
                "top_level_keys": top_level_keys,
                "value_type": value_type,
                "value_keys_preview": value_keys,
                "value_sample": value_sample,
            }
        )

    sample_doc_summary = pd.DataFrame(sample_rows)
    display(sample_doc_summary)

    print("\n=== Design C MongoDB Interpretation Help ===")
    print("Look for document types whose value contains minute-level timestamps or lists of intraday values.")
    print("Daily summary types at 00:00 only are not enough for strict bedtime cutoff.")

else:
    print("\n=== Raw File Fallback Check ===")
    raw_candidates = []
    for path in DATA_DIR.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()
        path_lower = str(path).lower()

        if suffix in [".bson", ".json", ".jsonl"] and any(
            token in path_lower
            for token in ["fitbit", "step", "calorie", "activity", "intraday", "minute"]
        ):
            raw_candidates.append(
                {
                    "relative_path": str(path.relative_to(PROJECT_ROOT)),
                    "suffix": suffix,
                    "size_mb": path.stat().st_size / (1024 ** 2),
                }
            )

    raw_candidate_df = pd.DataFrame(raw_candidates).sort_values(
        ["suffix", "relative_path"]
    ) if raw_candidates else pd.DataFrame(
        columns=["relative_path", "suffix", "size_mb"]
    )

    display(raw_candidate_df)

    print("\nMongoDB is not available from this notebook session.")
    print("If raw BSON exists, inspect document types before loading large collections.")

=== MongoDB Intraday Source Check ===
MongoDB available: True

=== Collection Basic Counts ===
estimated fitbit documents: 71284346

=== Fitbit Type Counts Containing Activity/Step/Calorie/Intraday Keywords ===


,fitbit_type,documents
0,heart_rate,48720040
1,calories,9675782
2,steps,3010529
3,Heart Rate Variability Details,220512
4,resting_heart_rate,12362
5,very_active_minutes,7203
6,lightly_active_minutes,7203
7,moderately_active_minutes,7203
8,sedentary_minutes,7203
9,time_in_heart_rate_zones,4876



=== Sample Documents For Candidate Types ===


,fitbit_type,top_level_keys,value_type,value_keys_preview,value_sample
0,heart_rate,"[_id, data, id, type]",NoneType,[],None
1,calories,"[_id, data, id, type]",NoneType,[],None
2,steps,"[_id, data, id, type]",NoneType,[],None
3,Heart Rate Variability Details,"[_id, data, id, type]",NoneType,[],None
4,resting_heart_rate,"[_id, data, id, type]",NoneType,[],None
5,very_active_minutes,"[_id, data, id, type]",NoneType,[],None
6,lightly_active_minutes,"[_id, data, id, type]",NoneType,[],None
7,moderately_active_minutes,"[_id, data, id, type]",NoneType,[],None
8,sedentary_minutes,"[_id, data, id, type]",NoneType,[],None
9,time_in_heart_rate_zones,"[_id, data, id, type]",NoneType,[],None



=== Design C MongoDB Interpretation Help ===
Look for document types whose value contains minute-level timestamps or lists of intraday values.
Daily summary types at 00:00 only are not enough for strict bedtime cutoff.


In [7]:
# Cell 7. MongoDB intraday candidate document data 구조 확인
# 원하는 결과:
# - steps/calories/heart_rate 문서의 data 필드 구조를 확인한다.
# - timestamp/value가 어디에 있는지 찾는다.
# - participant id가 어떤 필드에 있는지 확인한다.
# - 전체 collection을 로드하지 않는다.

if not mongo_available:
    raise RuntimeError("MongoDB is not available. Run Cell 6 first and check connection.")

candidate_intraday_types = ["steps", "calories", "heart_rate"]

sample_docs = []
for fitbit_type in candidate_intraday_types:
    cursor = collection.find({"type": fitbit_type}).limit(5)

    for doc_idx, doc in enumerate(cursor):
        data = doc.get("data", None)

        if isinstance(data, dict):
            data_type = "dict"
            data_keys = sorted(list(data.keys()))
            data_preview = {
                key: data.get(key)
                for key in data_keys[:12]
            }
        elif isinstance(data, list):
            data_type = "list"
            data_keys = []
            data_preview = data[:3]
        else:
            data_type = type(data).__name__
            data_keys = []
            data_preview = data

        sample_docs.append(
            {
                "fitbit_type": fitbit_type,
                "sample_index": doc_idx,
                "top_level_keys": sorted(list(doc.keys())),
                "id_field": doc.get("id"),
                "data_type": data_type,
                "data_keys": data_keys[:30],
                "data_preview": str(data_preview)[:1000],
            }
        )

sample_doc_detail = pd.DataFrame(sample_docs)

print("=== Intraday Candidate Document Data Structure ===")
display(sample_doc_detail)

# data 내부 key 빈도를 조금 더 확인합니다.
key_rows = []

for fitbit_type in candidate_intraday_types:
    cursor = collection.find({"type": fitbit_type}).limit(100)

    for doc in cursor:
        data = doc.get("data", None)

        if isinstance(data, dict):
            for key, value in data.items():
                key_rows.append(
                    {
                        "fitbit_type": fitbit_type,
                        "data_key": key,
                        "value_type": type(value).__name__,
                        "value_sample": str(value)[:200],
                    }
                )

data_key_summary = pd.DataFrame(key_rows)

if not data_key_summary.empty:
    print("\n=== Data Key Summary ===")
    display(
        data_key_summary
        .groupby(["fitbit_type", "data_key", "value_type"])
        .size()
        .reset_index(name="sample_docs")
        .sort_values(["fitbit_type", "sample_docs"], ascending=[True, False])
    )

    print("\n=== Data Key Examples ===")
    display(
        data_key_summary
        .drop_duplicates(["fitbit_type", "data_key", "value_type"])
        .sort_values(["fitbit_type", "data_key"])
        .reset_index(drop=True)
    )

print("\n=== Participant Id Mapping Check ===")
# fitbit doc의 id가 participant_object_id와 직접 같은지, 혹은 다른 매핑이 필요한지 확인합니다.
sample_ids = []
for fitbit_type in candidate_intraday_types:
    ids = collection.distinct("id", {"type": fitbit_type})
    sample_ids.append(
        {
            "fitbit_type": fitbit_type,
            "distinct_id_count": len(ids),
            "sample_ids": ids[:5],
        }
    )

display(pd.DataFrame(sample_ids))

known_participant_ids = set(sleep_target[ID_COL].astype(str).unique())
for row in sample_ids:
    overlap = set(map(str, row["sample_ids"])) & known_participant_ids
    print(row["fitbit_type"], "sample id overlap with participant_object_id:", overlap)

print("\nNext step:")
print("- If data has timestamp/dateTime and value fields, build a small pre-sleep cutoff aggregation prototype for steps/calories.")
print("- If id is not participant_object_id, find participant id mapping from extracted CSV or MongoDB fields.")

=== Intraday Candidate Document Data Structure ===


,fitbit_type,sample_index,top_level_keys,id_field,data_type,data_keys,data_preview
0,steps,0,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '2021-05-24T00:15:00', 'value': '0'}"
1,steps,1,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '2021-05-24T00:25:00', 'value': '0'}"
2,steps,2,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '2021-05-24T00:31:00', 'value': '0'}"
3,steps,3,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '2021-05-24T00:52:00', 'value': '7'}"
4,steps,4,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '2021-05-24T03:19:00', 'value': '0'}"
5,calories,0,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '05/24/21 00:00:00', 'value': '2.62'}"
6,calories,1,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '05/24/21 00:03:00', 'value': '1.09'}"
7,calories,2,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '05/24/21 00:06:00', 'value': '1.09'}"
8,calories,3,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '05/24/21 00:09:00', 'value': '1.09'}"
9,calories,4,"[_id, data, id, type]",621e2e8e67b776a24055b564,dict,"[dateTime, value]","{'dateTime': '05/24/21 00:12:00', 'value': '1.09'}"



=== Data Key Summary ===


,fitbit_type,data_key,value_type,sample_docs
0,calories,dateTime,str,100
1,calories,value,str,100
2,heart_rate,dateTime,str,100
3,heart_rate,value,dict,100
4,steps,dateTime,str,100
5,steps,value,str,100



=== Data Key Examples ===


,fitbit_type,data_key,value_type,value_sample
0,calories,dateTime,str,05/24/21 00:00:00
1,calories,value,str,2.62
2,heart_rate,dateTime,str,2021-05-24T00:00:16
3,heart_rate,value,dict,"{'bpm': 71, 'confidence': 1}"
4,steps,dateTime,str,2021-05-24T00:15:00
5,steps,value,str,0



=== Participant Id Mapping Check ===


,fitbit_type,distinct_id_count,sample_ids
0,steps,71,"[621e2e8e67b776a24055b564, 621e2eaf67b776a2406b14ac, 621e2ed667b776a24085d8d1, 621e2ef567b776a24099f889, 621e2efa67b776a2409dd1c3]"
1,calories,71,"[621e2e8e67b776a24055b564, 621e2eaf67b776a2406b14ac, 621e2ed667b776a24085d8d1, 621e2ef567b776a24099f889, 621e2efa67b776a2409dd1c3]"
2,heart_rate,71,"[621e2e8e67b776a24055b564, 621e2eaf67b776a2406b14ac, 621e2ed667b776a24085d8d1, 621e2ef567b776a24099f889, 621e2efa67b776a2409dd1c3]"


steps sample id overlap with participant_object_id: {'621e2e8e67b776a24055b564', '621e2efa67b776a2409dd1c3', '621e2eaf67b776a2406b14ac', '621e2ed667b776a24085d8d1', '621e2ef567b776a24099f889'}
calories sample id overlap with participant_object_id: {'621e2e8e67b776a24055b564', '621e2efa67b776a2409dd1c3', '621e2eaf67b776a2406b14ac', '621e2ed667b776a24085d8d1', '621e2ef567b776a24099f889'}
heart_rate sample id overlap with participant_object_id: {'621e2e8e67b776a24055b564', '621e2efa67b776a2409dd1c3', '621e2eaf67b776a2406b14ac', '621e2ed667b776a24085d8d1', '621e2ef567b776a24099f889'}

Next step:
- If data has timestamp/dateTime and value fields, build a small pre-sleep cutoff aggregation prototype for steps/calories.
- If id is not participant_object_id, find participant id mapping from extracted CSV or MongoDB fields.


In [8]:
# Cell 8. Design C prototype: sleep_start_datetime 이전 intraday 집계 테스트
# 원하는 결과:
# - 일부 sleep episode에 대해 startTime 이전 당일 intraday steps/calories/heart_rate를 집계한다.
# - MongoDB query 조건과 aggregation 로직이 맞는지 확인한다.
# - 전체 collection을 로드하지 않는다.
# - 아직 파일 저장은 하지 않는다.

from datetime import datetime, timedelta

def parse_mongo_datetime_series(value):
    """MongoDB Fitbit dateTime 문자열 포맷이 섞여 있어 pandas로 안전하게 파싱한다."""
    return pd.to_datetime(value, errors="coerce")


def query_intraday_docs(participant_id, fitbit_type, start_dt, end_dt):
    """
    participant_id와 type으로 후보 문서를 좁힌 뒤 Python에서 datetime cutoff를 적용한다.
    dateTime 문자열 포맷이 type마다 달라 MongoDB string range query는 아직 쓰지 않는다.
    prototype 단계라 하루 범위만 읽는다.
    """
    start_date = pd.Timestamp(start_dt).normalize()
    end_date = pd.Timestamp(end_dt).normalize()

    # dateTime 문자열 prefix가 포맷별로 달라서 넓게 participant/type으로 가져온 뒤 필터한다.
    # 성능은 prototype용이며, 전체 생성 단계에서는 type별 date string 조건을 추가해 최적화한다.
    cursor = collection.find(
        {
            "id": str(participant_id),
            "type": fitbit_type,
        },
        {
            "_id": 0,
            "id": 1,
            "type": 1,
            "data.dateTime": 1,
            "data.value": 1,
        },
    )

    rows = []
    for doc in cursor:
        data = doc.get("data", {})
        date_time_raw = data.get("dateTime")
        timestamp = pd.to_datetime(date_time_raw, errors="coerce")

        if pd.isna(timestamp):
            continue

        if not (pd.Timestamp(start_dt) <= timestamp < pd.Timestamp(end_dt)):
            continue

        value = data.get("value")

        if fitbit_type in ["steps", "calories"]:
            numeric_value = pd.to_numeric(value, errors="coerce")
            rows.append(
                {
                    "timestamp": timestamp,
                    "value": numeric_value,
                }
            )
        elif fitbit_type == "heart_rate":
            if isinstance(value, dict):
                bpm = pd.to_numeric(value.get("bpm"), errors="coerce")
                confidence = pd.to_numeric(value.get("confidence"), errors="coerce")
            else:
                bpm = np.nan
                confidence = np.nan

            rows.append(
                {
                    "timestamp": timestamp,
                    "bpm": bpm,
                    "confidence": confidence,
                }
            )

    return pd.DataFrame(rows)


def aggregate_pre_sleep_intraday_for_episode(row):
    participant_id = row[ID_COL]
    sleep_start = pd.Timestamp(row["startTime"])
    window_start = sleep_start.normalize()
    window_end = sleep_start

    steps_df = query_intraday_docs(participant_id, "steps", window_start, window_end)
    calories_df = query_intraday_docs(participant_id, "calories", window_start, window_end)
    heart_df = query_intraday_docs(participant_id, "heart_rate", window_start, window_end)

    result = {
        ID_COL: participant_id,
        "sleep_episode_id": row["sleep_episode_id"],
        "sleep_start_datetime": sleep_start,
        "pre_sleep_window_start": window_start,
        "pre_sleep_window_end": window_end,
        "pre_sleep_window_hours": (window_end - window_start).total_seconds() / 3600,
        TARGET_COL: row[TARGET_COL],
        "steps_pre_sleep_sum": steps_df["value"].sum() if not steps_df.empty else np.nan,
        "steps_pre_sleep_record_count": len(steps_df),
        "calories_pre_sleep_sum": calories_df["value"].sum() if not calories_df.empty else np.nan,
        "calories_pre_sleep_record_count": len(calories_df),
        "heart_rate_pre_sleep_mean": heart_df["bpm"].mean() if not heart_df.empty else np.nan,
        "heart_rate_pre_sleep_std": heart_df["bpm"].std() if len(heart_df) > 1 else 0.0 if len(heart_df) == 1 else np.nan,
        "heart_rate_pre_sleep_min": heart_df["bpm"].min() if not heart_df.empty else np.nan,
        "heart_rate_pre_sleep_max": heart_df["bpm"].max() if not heart_df.empty else np.nan,
        "heart_rate_pre_sleep_record_count": len(heart_df),
        "heart_rate_pre_sleep_mean_confidence": heart_df["confidence"].mean() if not heart_df.empty else np.nan,
    }

    return result, {
        "steps": steps_df,
        "calories": calories_df,
        "heart_rate": heart_df,
    }


# prototype 대상: 한 participant의 다양한 startTime을 포함하도록 앞쪽 5개 episode 사용
prototype_episodes = (
    episode_index
    .sort_values([ID_COL, "startTime"])
    .groupby(ID_COL)
    .head(5)
    .head(10)
    .copy()
)

prototype_rows = []
prototype_raw_examples = {}

for _, episode_row in prototype_episodes.iterrows():
    aggregated, raw_tables = aggregate_pre_sleep_intraday_for_episode(episode_row)
    prototype_rows.append(aggregated)

    if len(prototype_raw_examples) < 3:
        prototype_raw_examples[episode_row["sleep_episode_id"]] = raw_tables

prototype_aggregation = pd.DataFrame(prototype_rows)

print("=== Design C Prototype Episode Aggregation ===")
display(prototype_aggregation)

print("\n=== Prototype Missingness / Record Counts ===")
prototype_quality = pd.DataFrame(
    {
        "metric": [
            "episodes",
            "steps_missing_rows",
            "calories_missing_rows",
            "heart_rate_missing_rows",
            "mean_steps_records",
            "mean_calories_records",
            "mean_heart_rate_records",
        ],
        "value": [
            len(prototype_aggregation),
            int(prototype_aggregation["steps_pre_sleep_sum"].isna().sum()),
            int(prototype_aggregation["calories_pre_sleep_sum"].isna().sum()),
            int(prototype_aggregation["heart_rate_pre_sleep_mean"].isna().sum()),
            float(prototype_aggregation["steps_pre_sleep_record_count"].mean()),
            float(prototype_aggregation["calories_pre_sleep_record_count"].mean()),
            float(prototype_aggregation["heart_rate_pre_sleep_record_count"].mean()),
        ],
    }
)
display(prototype_quality)

print("\n=== Raw Intraday Examples For First Prototype Episodes ===")
for episode_id, raw_tables in prototype_raw_examples.items():
    print("\nEpisode:", episode_id)
    for name, table in raw_tables.items():
        print(f"{name}: shape={table.shape}")
        display(table.head(5))

print("\n=== Design C Prototype Interpretation ===")
print("If record counts are nonzero and timestamps stop before sleep_start_datetime, strict pre-sleep same-day aggregation is feasible.")

=== Design C Prototype Episode Aggregation ===


,participant_object_id,sleep_episode_id,sleep_start_datetime,pre_sleep_window_start,pre_sleep_window_end,pre_sleep_window_hours,good_sleep_label,steps_pre_sleep_sum,steps_pre_sleep_record_count,calories_pre_sleep_sum,calories_pre_sleep_record_count,heart_rate_pre_sleep_mean,heart_rate_pre_sleep_std,heart_rate_pre_sleep_min,heart_rate_pre_sleep_max,heart_rate_pre_sleep_record_count,heart_rate_pre_sleep_mean_confidence
0,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210524004000__20210524092100,2021-05-24 00:40:00,2021-05-24,2021-05-24 00:40:00,0.666667,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
1,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210524234830__20210525085630,2021-05-24 23:48:30,2021-05-24,2021-05-24 23:48:30,23.808333,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
2,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210525234630__20210526090630,2021-05-25 23:46:30,2021-05-25,2021-05-25 23:46:30,23.775000,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
3,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210526232130__20210527094830,2021-05-26 23:21:30,2021-05-26,2021-05-26 23:21:30,23.358333,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
4,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210527233700__20210528085830,2021-05-27 23:37:00,2021-05-27,2021-05-27 23:37:00,23.616667,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
5,621e2eaf67b776a2406b14ac,621e2eaf67b776a2406b14ac__20211028221500__20211029075300,2021-10-28 22:15:00,2021-10-28,2021-10-28 22:15:00,22.250000,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
6,621e2eaf67b776a2406b14ac,621e2eaf67b776a2406b14ac__20211029234700__20211030065143,2021-10-29 23:47:00,2021-10-29,2021-10-29 23:47:00,23.783333,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
7,621e2eaf67b776a2406b14ac,621e2eaf67b776a2406b14ac__20211031112430__20211031171500,2021-10-31 11:24:30,2021-10-31,2021-10-31 11:24:30,11.408333,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
8,621e2eaf67b776a2406b14ac,621e2eaf67b776a2406b14ac__20211101225200__20211102071830,2021-11-01 22:52:00,2021-11-01,2021-11-01 22:52:00,22.866667,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN
9,621e2eaf67b776a2406b14ac,621e2eaf67b776a2406b14ac__20211102214400__20211103065430,2021-11-02 21:44:00,2021-11-02,2021-11-02 21:44:00,21.733333,1,NaN,0,NaN,0,NaN,NaN,NaN,NaN,0,NaN



=== Prototype Missingness / Record Counts ===


,metric,value
0,episodes,10.0
1,steps_missing_rows,10.0
2,calories_missing_rows,10.0
3,heart_rate_missing_rows,10.0
4,mean_steps_records,0.0
5,mean_calories_records,0.0
6,mean_heart_rate_records,0.0



=== Raw Intraday Examples For First Prototype Episodes ===

Episode: 621e2e8e67b776a24055b564__20210524004000__20210524092100
steps: shape=(0, 0)


...

calories: shape=(0, 0)


""


heart_rate: shape=(0, 0)


""



=== Design C Prototype Interpretation ===
If record counts are nonzero and timestamps stop before sleep_start_datetime, strict pre-sleep same-day aggregation is feasible.


In [10]:
# Cell 8b. Design C prototype 최적화: MongoDB data.dateTime 범위 조건 사용
# 원하는 결과:
# - MongoDB fitbit.id 타입을 확인한다.
# - participant_object_id를 MongoDB id 타입에 맞춰 변환한다.
# - data.dateTime 범위 조건을 MongoDB query에 직접 넣어 빠르게 조회한다.
# - 일부 sleep episode에 대해 startTime 이전 당일 intraday steps/calories/heart_rate를 집계한다.
# - 아직 파일 저장은 하지 않는다.

from bson import ObjectId

def resolve_mongo_participant_id(participant_id):
    """
    sleep_target의 participant_object_id는 문자열이지만,
    MongoDB fitbit.id는 ObjectId일 수 있다.
    먼저 ObjectId로 조회해보고, 없으면 문자열로 조회한다.
    """
    participant_id_str = str(participant_id)

    if ObjectId.is_valid(participant_id_str):
        object_id_candidate = ObjectId(participant_id_str)

        exists_as_object_id = collection.find_one(
            {"id": object_id_candidate},
            {"_id": 1},
        )
        if exists_as_object_id is not None:
            return object_id_candidate

    exists_as_string = collection.find_one(
        {"id": participant_id_str},
        {"_id": 1},
    )
    if exists_as_string is not None:
        return participant_id_str

    return participant_id_str


def build_datetime_filter_for_type(fitbit_type, start_dt, end_dt):
    """
    MongoDB data.dateTime은 type별 문자열 포맷이 다르다.

    - steps, heart_rate:
      예: 2021-05-24T00:15:00

    - calories:
      예: 05/24/21 00:00:00

    문자열 range query가 가능하도록 type별 포맷으로 변환한다.
    """
    start_ts = pd.Timestamp(start_dt)
    end_ts = pd.Timestamp(end_dt)

    if fitbit_type in ["steps", "heart_rate"]:
        return {
            "$gte": start_ts.strftime("%Y-%m-%dT%H:%M:%S"),
            "$lt": end_ts.strftime("%Y-%m-%dT%H:%M:%S"),
        }

    if fitbit_type == "calories":
        return {
            "$gte": start_ts.strftime("%m/%d/%y %H:%M:%S"),
            "$lt": end_ts.strftime("%m/%d/%y %H:%M:%S"),
        }

    raise ValueError(f"Unsupported fitbit_type: {fitbit_type}")


print("=== MongoDB id Type Check ===")
for fitbit_type in ["steps", "calories", "heart_rate"]:
    doc = collection.find_one({"type": fitbit_type})
    print(
        fitbit_type,
        "id value:",
        doc.get("id"),
        "| python type:",
        type(doc.get("id")).__name__,
        "| data.dateTime example:",
        doc.get("data", {}).get("dateTime"),
    )


def query_intraday_docs_v2(participant_id, fitbit_type, start_dt, end_dt):
    mongo_id = resolve_mongo_participant_id(participant_id)
    datetime_filter = build_datetime_filter_for_type(fitbit_type, start_dt, end_dt)

    cursor = collection.find(
        {
            "id": mongo_id,
            "type": fitbit_type,
            "data.dateTime": datetime_filter,
        },
        {
            "_id": 0,
            "id": 1,
            "type": 1,
            "data.dateTime": 1,
            "data.value": 1,
        },
    )

    rows = []
    start_ts = pd.Timestamp(start_dt)
    end_ts = pd.Timestamp(end_dt)

    for doc in cursor:
        data = doc.get("data", {})
        date_time_raw = data.get("dateTime")
        timestamp = pd.to_datetime(date_time_raw, errors="coerce")

        if pd.isna(timestamp):
            continue

        if getattr(timestamp, "tzinfo", None) is not None:
            timestamp = timestamp.tz_localize(None)

        # MongoDB string range로 대부분 걸렀지만, 안전하게 한 번 더 확인한다.
        if not (start_ts <= timestamp < end_ts):
            continue

        value = data.get("value")

        if fitbit_type in ["steps", "calories"]:
            numeric_value = pd.to_numeric(value, errors="coerce")
            rows.append(
                {
                    "timestamp": timestamp,
                    "value": numeric_value,
                }
            )

        elif fitbit_type == "heart_rate":
            if isinstance(value, dict):
                bpm = pd.to_numeric(value.get("bpm"), errors="coerce")
                confidence = pd.to_numeric(value.get("confidence"), errors="coerce")
            else:
                bpm = np.nan
                confidence = np.nan

            rows.append(
                {
                    "timestamp": timestamp,
                    "bpm": bpm,
                    "confidence": confidence,
                }
            )

    return pd.DataFrame(rows)


def aggregate_pre_sleep_intraday_for_episode_v2(row):
    participant_id = row[ID_COL]
    sleep_start = pd.Timestamp(row["startTime"])
    window_start = sleep_start.normalize()
    window_end = sleep_start

    steps_df = query_intraday_docs_v2(participant_id, "steps", window_start, window_end)
    calories_df = query_intraday_docs_v2(participant_id, "calories", window_start, window_end)
    heart_df = query_intraday_docs_v2(participant_id, "heart_rate", window_start, window_end)

    result = {
        ID_COL: participant_id,
        "mongo_id_type": type(resolve_mongo_participant_id(participant_id)).__name__,
        "sleep_episode_id": row["sleep_episode_id"],
        "sleep_start_datetime": sleep_start,
        "pre_sleep_window_start": window_start,
        "pre_sleep_window_end": window_end,
        "pre_sleep_window_hours": (window_end - window_start).total_seconds() / 3600,
        TARGET_COL: row[TARGET_COL],

        "steps_pre_sleep_sum": steps_df["value"].sum() if not steps_df.empty else np.nan,
        "steps_pre_sleep_record_count": len(steps_df),

        "calories_pre_sleep_sum": calories_df["value"].sum() if not calories_df.empty else np.nan,
        "calories_pre_sleep_record_count": len(calories_df),

        "heart_rate_pre_sleep_mean": heart_df["bpm"].mean() if not heart_df.empty else np.nan,
        "heart_rate_pre_sleep_std": heart_df["bpm"].std() if len(heart_df) > 1 else 0.0 if len(heart_df) == 1 else np.nan,
        "heart_rate_pre_sleep_min": heart_df["bpm"].min() if not heart_df.empty else np.nan,
        "heart_rate_pre_sleep_max": heart_df["bpm"].max() if not heart_df.empty else np.nan,
        "heart_rate_pre_sleep_record_count": len(heart_df),
        "heart_rate_pre_sleep_mean_confidence": heart_df["confidence"].mean() if not heart_df.empty else np.nan,
    }

    return result, {
        "steps": steps_df,
        "calories": calories_df,
        "heart_rate": heart_df,
    }


# prototype 대상:
# 너무 이른 새벽 수면은 pre-sleep window가 짧으므로,
# 먼저 밤 21시 이후 시작 episode 3개만 확인한다.
prototype_episodes_v2 = (
    episode_index[
        episode_index["startTime"].dt.hour >= 21
    ]
    .sort_values([ID_COL, "startTime"])
    .groupby(ID_COL)
    .head(3)
    .head(3)
    .copy()
)

print("\n=== Prototype Episodes V2 ===")
display(
    prototype_episodes_v2[
        [
            ID_COL,
            "sleep_episode_id",
            "startTime",
            "endTime",
            "sleep_start_date",
            "sleep_end_date",
            TARGET_COL,
        ]
    ]
)

prototype_rows_v2 = []
prototype_raw_examples_v2 = {}

for _, episode_row in prototype_episodes_v2.iterrows():
    aggregated, raw_tables = aggregate_pre_sleep_intraday_for_episode_v2(episode_row)
    prototype_rows_v2.append(aggregated)

    if len(prototype_raw_examples_v2) < 3:
        prototype_raw_examples_v2[episode_row["sleep_episode_id"]] = raw_tables

prototype_aggregation_v2 = pd.DataFrame(prototype_rows_v2)

print("\n=== Design C Prototype Episode Aggregation V2 ===")
display(prototype_aggregation_v2)

print("\n=== Prototype V2 Missingness / Record Counts ===")
prototype_quality_v2 = pd.DataFrame(
    {
        "metric": [
            "episodes",
            "steps_missing_rows",
            "calories_missing_rows",
            "heart_rate_missing_rows",
            "mean_steps_records",
            "mean_calories_records",
            "mean_heart_rate_records",
        ],
        "value": [
            len(prototype_aggregation_v2),
            int(prototype_aggregation_v2["steps_pre_sleep_sum"].isna().sum()),
            int(prototype_aggregation_v2["calories_pre_sleep_sum"].isna().sum()),
            int(prototype_aggregation_v2["heart_rate_pre_sleep_mean"].isna().sum()),
            float(prototype_aggregation_v2["steps_pre_sleep_record_count"].mean()),
            float(prototype_aggregation_v2["calories_pre_sleep_record_count"].mean()),
            float(prototype_aggregation_v2["heart_rate_pre_sleep_record_count"].mean()),
        ],
    }
)
display(prototype_quality_v2)

print("\n=== Raw Intraday Examples V2 ===")
for episode_id, raw_tables in prototype_raw_examples_v2.items():
    print("\nEpisode:", episode_id)
    for name, table in raw_tables.items():
        print(f"{name}: shape={table.shape}")
        display(table.head(5))
        display(table.tail(5))

print("\n=== Design C Prototype V2 Interpretation ===")
print("If record counts are nonzero, Design C is feasible with MongoDB date-range filtering.")

=== MongoDB id Type Check ===
steps id value: 621e2e8e67b776a24055b564 | python type: ObjectId | data.dateTime example: 2021-05-24T00:15:00
calories id value: 621e2e8e67b776a24055b564 | python type: ObjectId | data.dateTime example: 05/24/21 00:00:00
heart_rate id value: 621e2e8e67b776a24055b564 | python type: ObjectId | data.dateTime example: 2021-05-24T00:00:16

=== Prototype Episodes V2 ===


,participant_object_id,sleep_episode_id,startTime,endTime,sleep_start_date,sleep_end_date,good_sleep_label
1,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210524234830__20210525085630,2021-05-24 23:48:30,2021-05-25 08:56:30,2021-05-24,2021-05-25,1
2,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210525234630__20210526090630,2021-05-25 23:46:30,2021-05-26 09:06:30,2021-05-25,2021-05-26,1
3,621e2e8e67b776a24055b564,621e2e8e67b776a24055b564__20210526232130__20210527094830,2021-05-26 23:21:30,2021-05-27 09:48:30,2021-05-26,2021-05-27,1



=== Design C Prototype Episode Aggregation V2 ===


,participant_object_id,mongo_id_type,sleep_episode_id,sleep_start_datetime,pre_sleep_window_start,pre_sleep_window_end,pre_sleep_window_hours,good_sleep_label,steps_pre_sleep_sum,steps_pre_sleep_record_count,calories_pre_sleep_sum,calories_pre_sleep_record_count,heart_rate_pre_sleep_mean,heart_rate_pre_sleep_std,heart_rate_pre_sleep_min,heart_rate_pre_sleep_max,heart_rate_pre_sleep_record_count,heart_rate_pre_sleep_mean_confidence
0,621e2e8e67b776a24055b564,ObjectId,621e2e8e67b776a24055b564__20210524234830__20210525085630,2021-05-24 23:48:30,2021-05-24,2021-05-24 23:48:30,23.808333,1,8823,679,2335.76,1429,71.677430,13.469360,32,144,12481,1.813877
1,621e2e8e67b776a24055b564,ObjectId,621e2e8e67b776a24055b564__20210525234630__20210526090630,2021-05-25 23:46:30,2021-05-25,2021-05-25 23:46:30,23.775000,1,9720,672,2315.94,1427,70.633747,13.623156,32,143,12576,1.840808
2,621e2e8e67b776a24055b564,ObjectId,621e2e8e67b776a24055b564__20210526232130__20210527094830,2021-05-26 23:21:30,2021-05-26,2021-05-26 23:21:30,23.358333,1,8211,572,2214.86,1402,71.978559,14.140321,36,127,12173,1.774747



=== Prototype V2 Missingness / Record Counts ===


...

,timestamp,value
1397,2021-05-26 23:17:00,1.09
1398,2021-05-26 23:18:00,3.06
1399,2021-05-26 23:19:00,1.09
1400,2021-05-26 23:20:00,1.09
1401,2021-05-26 23:21:00,1.09


heart_rate: shape=(12173, 3)


,timestamp,bpm,confidence
0,2021-05-26 00:00:04,60,1
1,2021-05-26 00:00:09,59,2
2,2021-05-26 00:00:14,60,1
3,2021-05-26 00:00:24,60,3
4,2021-05-26 00:00:29,61,3


,timestamp,bpm,confidence
12168,2021-05-26 23:19:26,61,2
12169,2021-05-26 23:19:31,62,3
12170,2021-05-26 23:19:36,61,2
12171,2021-05-26 23:19:41,61,1
12172,2021-05-26 23:19:46,61,1



=== Design C Prototype V2 Interpretation ===
If record counts are nonzero, Design C is feasible with MongoDB date-range filtering.


In [11]:
# Cell 9. Design C 최종 feature specification 초안
# 원하는 결과:
# - strict pre-sleep forecasting dataset에서 만들 feature 목록을 정의한다.
# - MongoDB intraday source와 daily/history source를 구분한다.
# - leakage risk와 inclusion decision을 명확히 한다.
# - 아직 파일 저장은 하지 않는다.

design_c_feature_spec_rows = [
    {
        "feature_group": "pre_sleep_steps",
        "source": "MongoDB fitbit type=steps",
        "time_window": "sleep_start_date 00:00 <= timestamp < sleep_start_datetime",
        "features": [
            "steps_pre_sleep_sum",
            "steps_pre_sleep_record_count",
            "steps_pre_sleep_active_record_count",
            "steps_pre_sleep_last_3h_sum",
            "steps_pre_sleep_last_1h_sum",
        ],
        "include_in_first_dataset": True,
        "leakage_risk": "low",
        "notes": "Strictly before sleep onset.",
    },
    {
        "feature_group": "pre_sleep_calories",
        "source": "MongoDB fitbit type=calories",
        "time_window": "sleep_start_date 00:00 <= timestamp < sleep_start_datetime",
        "features": [
            "calories_pre_sleep_sum",
            "calories_pre_sleep_record_count",
            "calories_pre_sleep_last_3h_sum",
            "calories_pre_sleep_last_1h_sum",
        ],
        "include_in_first_dataset": True,
        "leakage_risk": "low",
        "notes": "Strictly before sleep onset.",
    },
    {
        "feature_group": "pre_sleep_heart_rate",
        "source": "MongoDB fitbit type=heart_rate",
        "time_window": "sleep_start_date 00:00 <= timestamp < sleep_start_datetime",
        "features": [
            "heart_rate_pre_sleep_mean",
            "heart_rate_pre_sleep_std",
            "heart_rate_pre_sleep_min",
            "heart_rate_pre_sleep_max",
            "heart_rate_pre_sleep_median",
            "heart_rate_pre_sleep_last_3h_mean",
            "heart_rate_pre_sleep_last_1h_mean",
            "heart_rate_pre_sleep_record_count",
            "heart_rate_pre_sleep_mean_confidence",
        ],
        "include_in_first_dataset": True,
        "leakage_risk": "low",
        "notes": "Strictly before sleep onset. Confidence can be used for quality control.",
    },
    {
        "feature_group": "pre_sleep_window_timing",
        "source": "sleep_daily_target.csv startTime",
        "time_window": "known at prediction time",
        "features": [
            "pre_sleep_window_hours",
            "sleep_start_hour",
            "sleep_start_dayofweek_sin",
            "sleep_start_dayofweek_cos",
            "sleep_start_month_sin",
            "sleep_start_month_cos",
        ],
        "include_in_first_dataset": True,
        "leakage_risk": "low",
        "notes": "Calendar/time features are known before prediction.",
    },
    {
        "feature_group": "previous_day_daily_activity",
        "source": "processed daily aggregates",
        "time_window": "sleep_start_date - 1 day",
        "features": [
            "previous_day_steps_sum",
            "previous_day_calories_sum",
            "previous_day_lightly_active_minutes_sum",
            "previous_day_moderately_active_minutes_sum",
            "previous_day_sedentary_minutes_sum",
            "previous_day_very_active_minutes_sum",
            "previous_day_resting_hr_mean",
        ],
        "include_in_first_dataset": True,
        "leakage_risk": "low",
        "notes": "Already available before sleep onset.",
    },
    {
        "feature_group": "rolling_history",
        "source": "Design C pre-sleep features and previous-day daily features",
        "time_window": "previous 3/7 available days before current sleep episode",
        "features": [
            "rolling_3d_mean/std/min/max",
            "rolling_7d_mean/std/min/max",
            "diff1",
            "dev_from_rolling_mean",
        ],
        "include_in_first_dataset": True,
        "leakage_risk": "low",
        "notes": "Must not include the target sleep episode outcome.",
    },
    {
        "feature_group": "target_sleep_outcome_features",
        "source": "target sleep episode",
        "time_window": "after sleep onset/end",
        "features": [
            "minutesAsleep",
            "timeInBed",
            "sleep_duration_hours",
            "efficiency",
            "sleep stages",
            "target sleep respiratory",
            "target sleep HRV",
            "target sleep SpO2",
        ],
        "include_in_first_dataset": False,
        "leakage_risk": "high",
        "notes": "Forbidden as predictors; target/label derivation only.",
    },
    {
        "feature_group": "same_day_daily_aggregate",
        "source": "processed daily aggregates",
        "time_window": "full sleep_start_date",
        "features": [
            "full_day_steps_sum",
            "full_day_calories_sum",
            "full_day_activity_minutes",
        ],
        "include_in_first_dataset": False,
        "leakage_risk": "medium",
        "notes": "Use only as Design B approximation, not strict Design C.",
    },
]

design_c_feature_spec = pd.DataFrame(design_c_feature_spec_rows)

print("=== Design C Feature Specification ===")
display(design_c_feature_spec)

print("\n=== Design C First Dataset Included Groups ===")
display(
    design_c_feature_spec[
        design_c_feature_spec["include_in_first_dataset"]
    ][
        [
            "feature_group",
            "source",
            "time_window",
            "leakage_risk",
            "notes",
        ]
    ]
)

print("\n=== Design C Excluded / Comparison-Only Groups ===")
display(
    design_c_feature_spec[
        ~design_c_feature_spec["include_in_first_dataset"]
    ][
        [
            "feature_group",
            "source",
            "time_window",
            "leakage_risk",
            "notes",
        ]
    ]
)

print("\n=== Recommended Next Notebook ===")
print("notebooks/11_create_pre_sleep_forecasting_dataset.ipynb")
print("Goal: create strict Design C dataset from MongoDB intraday steps/calories/heart_rate plus previous-day/rolling features.")

=== Design C Feature Specification ===


,feature_group,source,time_window,features,include_in_first_dataset,leakage_risk,notes
0,pre_sleep_steps,MongoDB fitbit type=steps,sleep_start_date 00:00 <= timestamp < sleep_start_datetime,"[steps_pre_sleep_sum, steps_pre_sleep_record_count, steps_pre_sleep_active_record_count, steps_pre_sleep_last_3h_sum, steps_pre_sleep_last_1h_sum]",True,low,Strictly before sleep onset.
1,pre_sleep_calories,MongoDB fitbit type=calories,sleep_start_date 00:00 <= timestamp < sleep_start_datetime,"[calories_pre_sleep_sum, calories_pre_sleep_record_count, calories_pre_sleep_last_3h_sum, calories_pre_sleep_last_1h_sum]",True,low,Strictly before sleep onset.
2,pre_sleep_heart_rate,MongoDB fitbit type=heart_rate,sleep_start_date 00:00 <= timestamp < sleep_start_datetime,"[heart_rate_pre_sleep_mean, heart_rate_pre_sleep_std, heart_rate_pre_sleep_min, heart_rate_pre_sleep_max, heart_rate_pre_sleep_median, heart_rate_pre_sleep_...",True,low,Strictly before sleep onset. Confidence can be used for quality control.
3,pre_sleep_window_timing,sleep_daily_target.csv startTime,known at prediction time,"[pre_sleep_window_hours, sleep_start_hour, sleep_start_dayofweek_sin, sleep_start_dayofweek_cos, sleep_start_month_sin, sleep_start_month_cos]",True,low,Calendar/time features are known before prediction.
4,previous_day_daily_activity,processed daily aggregates,sleep_start_date - 1 day,"[previous_day_steps_sum, previous_day_calories_sum, previous_day_lightly_active_minutes_sum, previous_day_moderately_active_minutes_sum, previous_day_sedent...",True,low,Already available before sleep onset.
5,rolling_history,Design C pre-sleep features and previous-day daily features,previous 3/7 available days before current sleep episode,"[rolling_3d_mean/std/min/max, rolling_7d_mean/std/min/max, diff1, dev_from_rolling_mean]",True,low,Must not include the target sleep episode outcome.
6,target_sleep_outcome_features,target sleep episode,after sleep onset/end,"[minutesAsleep, timeInBed, sleep_duration_hours, efficiency, sleep stages, target sleep respiratory, target sleep HRV, target sleep SpO2]",False,high,Forbidden as predictors; target/label derivation only.
7,same_day_daily_aggregate,processed daily aggregates,full sleep_start_date,"[full_day_steps_sum, full_day_calories_sum, full_day_activity_minutes]",False,medium,"Use only as Design B approximation, not strict Design C."



=== Design C First Dataset Included Groups ===


,feature_group,source,time_window,leakage_risk,notes
0,pre_sleep_steps,MongoDB fitbit type=steps,sleep_start_date 00:00 <= timestamp < sleep_start_datetime,low,Strictly before sleep onset.
1,pre_sleep_calories,MongoDB fitbit type=calories,sleep_start_date 00:00 <= timestamp < sleep_start_datetime,low,Strictly before sleep onset.
2,pre_sleep_heart_rate,MongoDB fitbit type=heart_rate,sleep_start_date 00:00 <= timestamp < sleep_start_datetime,low,Strictly before sleep onset. Confidence can be used for quality control.
3,pre_sleep_window_timing,sleep_daily_target.csv startTime,known at prediction time,low,Calendar/time features are known before prediction.
4,previous_day_daily_activity,processed daily aggregates,sleep_start_date - 1 day,low,Already available before sleep onset.
5,rolling_history,Design C pre-sleep features and previous-day daily features,previous 3/7 available days before current sleep episode,low,Must not include the target sleep episode outcome.



=== Design C Excluded / Comparison-Only Groups ===


,feature_group,source,time_window,leakage_risk,notes
6,target_sleep_outcome_features,target sleep episode,after sleep onset/end,high,Forbidden as predictors; target/label derivation only.
7,same_day_daily_aggregate,processed daily aggregates,full sleep_start_date,medium,"Use only as Design B approximation, not strict Design C."



=== Recommended Next Notebook ===
notebooks/11_create_pre_sleep_forecasting_dataset.ipynb
Goal: create strict Design C dataset from MongoDB intraday steps/calories/heart_rate plus previous-day/rolling features.


In [12]:
# Cell 10. pre-sleep forecasting dataset design report 저장 및 log 업데이트
# 원하는 결과:
# - Design C를 최종 선택안으로 기록한다.
# - feature specification CSV와 설계 보고서를 저장한다.
# - log/2026-06-29.md를 업데이트한다.
# - training이나 full dataset 생성은 하지 않는다.

from datetime import datetime

PRE_SLEEP_DIR = PROCESSED_DIR / "pre_sleep_forecasting"
PRE_SLEEP_DIR.mkdir(parents=True, exist_ok=True)

DESIGN_REPORT_PATH = REPORTS_DIR / "pre_sleep_forecasting_dataset_design.md"
FEATURE_SPEC_PATH = PRE_SLEEP_DIR / "pre_sleep_design_c_feature_spec.csv"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

design_c_feature_spec.to_csv(FEATURE_SPEC_PATH, index=False, encoding="utf-8-sig")

report_text = f"""# Pre-Sleep Forecasting Dataset Design

## Purpose

The new prediction goal is:

`Use wearable/context data available before sleep onset on the same day, plus prior history, to predict the quality of the upcoming sleep episode.`

The prediction unit is a participant-level sleep episode.

## Key Alignment Rule

Prediction time:

`prediction_time = sleep_start_datetime`

Allowed predictors:

`feature_timestamp < sleep_start_datetime`

Forbidden predictors:

- target sleep episode outcomes
- sleep duration, time in bed, efficiency, sleep stages
- target sleep respiratory summaries
- target sleep HRV or SpO2
- any feature generated after sleep onset/end

## Source Findings

Sleep episode source files:

- `data/processed/daily_aggregates/sleep_daily_target.csv`
- `data/processed/modeling_dataset_daily.csv`

Both contain:

- `participant_object_id`
- `calendar_date`
- `startTime`
- `endTime`
- `good_sleep_label`

The existing `calendar_date` matches sleep end date for all rows.

| item | value |
| --- | ---: |
| sleep episodes | 3551 |
| participants | 69 |
| target mean | 0.3937 |
| calendar date matches sleep end date | 1.0000 |
| cross-midnight sleep ratio | 0.3799 |

This means the previous same-date experiments used sleep-end-date alignment and should not be interpreted as pre-sleep forecasting.

## Design Options

### Design A: Conservative Previous-Day History

Uses `sleep_start_date - 1` daily features.

- Low leakage risk
- Does not include same-day pre-sleep activity
- Useful as conservative baseline

### Design B: Sleep-Start-Date Daily Approximation

Uses daily aggregate features from `sleep_start_date`.

- Coverage: approximately 95.7%
- More aligned with pre-sleep prediction than sleep-end-date same-date features
- But full-day daily aggregates can include small post-bedtime activity
- Useful as approximation/comparison only

### Design C: Strict Intraday Pre-Sleep Aggregation

Uses MongoDB intraday Fitbit documents and cuts every feature at `sleep_start_datetime`.

Confirmed MongoDB source types:

| Fitbit type | timestamp field | value field | notes |
| --- | --- | --- | --- |
| `steps` | `data.dateTime` | `data.value` | intraday step counts |
| `calories` | `data.dateTime` | `data.value` | minute-level calories |
| `heart_rate` | `data.dateTime` | `data.value.bpm`, `data.value.confidence` | high-frequency heart rate |

Prototype aggregation confirmed nonzero records for pre-sleep windows and timestamps stopped before sleep onset.

## Selected Design

Selected design:

`Design C: strict intraday pre-sleep forecasting dataset`

Reason:

- It best matches the real prediction objective.
- It uses only data available before sleep onset.
- It avoids sleep-end-date leakage.
- MongoDB contains enough intraday steps, calories, and heart-rate data to build the dataset.

## Feature Specification

| feature group | source | time window | include first dataset | leakage risk |
| --- | --- | --- | --- | --- |
"""

for _, row in design_c_feature_spec.iterrows():
    report_text += (
        f"| `{row['feature_group']}` | {row['source']} | "
        f"{row['time_window']} | {row['include_in_first_dataset']} | {row['leakage_risk']} |\n"
    )

report_text += """

## Recommended Implementation Staging

### Stage 1

Create a strict Design C tabular dataset with:

- pre-sleep steps
- pre-sleep calories
- pre-sleep heart rate
- pre-sleep timing/calendar features
- previous-day daily activity/resting-HR features

### Stage 2

Add rolling/trend features after Stage 1 validation:

- 3-day rolling mean/std/min/max
- 7-day rolling mean/std/min/max
- 1-day difference
- deviation from rolling means

### Stage 3

Create tensors for:

- MLP current episode
- GRU window 7
- optional GRU window 14

## Next Notebook

Recommended next notebook:

`notebooks/11_create_pre_sleep_forecasting_dataset.ipynb`

Goal:

Create the strict Design C dataset from MongoDB intraday steps/calories/heart_rate plus previous-day daily features.

## Interpretation Boundary

The earlier best model remains useful as a same-date sleep classification result:

`phase1a_003 / same_date / daytime_only / window 7 / mlp_current_day`

However, the new pre-sleep forecasting objective requires a new sleep-start-aligned dataset. The earlier same-date model should not be used as evidence of strict pre-sleep forecasting performance.
"""

DESIGN_REPORT_PATH.write_text(report_text, encoding="utf-8")

log_entry = f"""

## Pre-Sleep Forecasting Dataset Design

### 1. Reason for update

- Started `notebooks/10_pre_sleep_forecasting_dataset_design.ipynb`.
- The prediction objective was redefined as pre-sleep forecasting:
  - use data available before `sleep_start_datetime`
  - predict the quality of the upcoming sleep episode
- This differs from previous same-date classification and previous-day timing sensitivity experiments.

### 2. Main findings

- Sleep episode source files contain `startTime`, `endTime`, and `good_sleep_label`:
  - `data/processed/daily_aggregates/sleep_daily_target.csv`
  - `data/processed/modeling_dataset_daily.csv`
- Existing `calendar_date` matches sleep end date for all rows.
- Cross-midnight sleep ratio is approximately `0.3799`.
- Therefore previous same-date results should not be interpreted as pre-sleep forecasting.
- MongoDB Fitbit intraday documents are available for:
  - `steps`
  - `calories`
  - `heart_rate`
- A prototype confirmed strict pre-sleep aggregation before `sleep_start_datetime` is feasible.

### 3. Created outputs

- Created:
  - `reports/pre_sleep_forecasting_dataset_design.md`
  - `data/processed/pre_sleep_forecasting/pre_sleep_design_c_feature_spec.csv`

### 4. Selected design

- Selected:
  - `Design C: strict intraday pre-sleep forecasting dataset`
- Reason:
  - best matches the intended prediction objective
  - uses only timestamps before sleep onset
  - avoids sleep-end-date same-date leakage

### 5. Next planned work

- Create:
  - `notebooks/11_create_pre_sleep_forecasting_dataset.ipynb`
- Goal:
  - build the strict Design C dataset from MongoDB intraday steps/calories/heart_rate and previous-day daily features.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("feature spec:", FEATURE_SPEC_PATH.relative_to(PROJECT_ROOT), FEATURE_SPEC_PATH.exists())
print("design report:", DESIGN_REPORT_PATH.relative_to(PROJECT_ROOT), DESIGN_REPORT_PATH.exists())
print("report characters:", len(report_text))
print("log updated:", LOG_PATH, LOG_PATH.exists())
print("log appended characters:", len(log_entry))

feature spec: data\processed\pre_sleep_forecasting\pre_sleep_design_c_feature_spec.csv True
design report: reports\pre_sleep_forecasting_dataset_design.md True
report characters: 5053
log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
log appended characters: 1681
